# AI Engineering Platform ? From-Scratch Language Model

??? ?????? ????? ???? ????? ????? Transformer ?? ????? ???????? ??? ????? ????? ?????? ?????? ?? ??????? ?????? ????? ???? ??????? ??????.

**??? ???????:** ?? ????? Colab ???? `Runtime ? Change runtime type ? T4 GPU` ?? ??? ??????. ???? ????? ???????? ??? CPU ?????? ???? ?? ???? ????.

**???:** ?????? ???????? ???????? ????? ?????? ?? ????? ?? ??????? ???? ??? ???? ??????? ????? ?? ???? ?????.

In [7]:
from pathlib import Path
import os
import subprocess
import sys

REPO = Path('/content/ai-engineering-platform')
if not REPO.exists():
    subprocess.run([
        'git', 'clone',
        'https://github.com/F13342/ai-engineering-platform.git',
        str(REPO),
    ], check=True)

os.chdir(REPO)
sys.path.insert(0, str(REPO / 'src'))

try:
    import torch
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch'], check=True)
    import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Project:', REPO)


PyTorch: 2.11.0+cpu
CUDA available: False
Project: /content/ai-engineering-platform


In [9]:
import os
import sys
import subprocess
from pathlib import Path

repo = Path("/content/ai-engineering-platform").resolve()
os.chdir(repo)

# تحديث نسخة المشروع من GitHub
pull = subprocess.run(
    ["git", "pull", "--ff-only", "origin", "main"],
    capture_output=True,
    text=True,
)
print(pull.stdout)
if pull.stderr:
    print(pull.stderr)
if pull.returncode != 0:
    raise RuntimeError("تعذر تحديث المشروع من GitHub.")

# تجهيز pytest
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pytest"],
    check=True,
)

# تمرير مسار src إلى عملية الاختبارات
env = os.environ.copy()
env["PYTHONPATH"] = (
    str(repo / "src") + os.pathsep + env.get("PYTHONPATH", "")
)

result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q", "-ra",
        "tests/test_language_model.py",
        "tests/test_language_model_tokenizer.py",
        "tests/test_language_model_training.py",
    ],
    env=env,
    capture_output=True,
    text=True,
)

print("\n--- TEST RESULTS ---")
print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("فشلت بعض الاختبارات؛ نحتاج إلى معالجة السبب.")

Updating bb7e43c..bcd3a6c
Fast-forward
 tests/test_language_model_training.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)

From https://github.com/F13342/ai-engineering-platform
 * branch            main       -> FETCH_HEAD
   bb7e43c..bcd3a6c  main       -> origin/main


--- TEST RESULTS ---
.................                                                        [100%]
17 passed in 4.61s



## 1. ????? ?????????? ???????

????? ??? ?????? ???????? ???????? ??????? ?????????? ????????? ???????? ?????? ?????? ???????. ??? ??? ??????? ????? ??????? ????? ??? ?????? ??????? ??????.

In [10]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest'], check=True)
subprocess.run([
    sys.executable, '-m', 'pytest', '-q',
    'tests/test_language_model.py',
    'tests/test_language_model_tokenizer.py',
    'tests/test_language_model_training.py',
], check=True)


CalledProcessError: Command '['/usr/bin/python3', '-m', 'pytest', '-q', 'tests/test_language_model.py', 'tests/test_language_model_tokenizer.py', 'tests/test_language_model_training.py']' returned non-zero exit status 2.

In [11]:
import os
import sys
from pathlib import Path

repo = Path("/content/ai-engineering-platform")
os.chdir(repo)
sys.path.insert(0, str(repo / "src"))

from ai_platform.machine_learning.language_model.training import (
    LanguageModelTrainingConfig,
    train_language_model,
)

# نص اصطناعي قصير لاختبار مسار التدريب فقط
corpus_text = (
    "AI engineering platform trains its own language model from random initialization. "
    "A tokenizer converts characters into token IDs. A transformer learns to predict "
    "the next character from previous characters. The training loop measures loss, "
    "updates model parameters, and saves checkpoints. "
) * 120

corpus_path = Path("/content/model_lab_smoke_corpus.txt")
corpus_path.write_text(corpus_text, encoding="utf-8")

output_dir = Path("/content/model_lab_smoke_artifacts")

config = LanguageModelTrainingConfig(
    block_size=32,
    n_layers=2,
    n_heads=2,
    n_embd=64,
    dropout=0.1,
    batch_size=16,
    max_steps=30,
    learning_rate=3e-4,
    eval_interval=10,
    eval_batches=3,
    train_fraction=0.9,
    seed=42,
    device="auto",
)

result = train_language_model(corpus_path, output_dir, config)

print("\nTraining result:")
print(result)

assert (output_dir / "best.pt").is_file()
assert (output_dir / "last.pt").is_file()
assert (output_dir / "tokenizer.json").is_file()
assert (output_dir / "metrics.json").is_file()

print("\nSUCCESS: Training and checkpoint creation completed.")


step 10/30 | train_loss=3.1730 | val_loss=3.1707 | device=cpu
step 20/30 | train_loss=3.0037 | val_loss=3.0093 | device=cpu
step 30/30 | train_loss=2.8625 | val_loss=2.8622 | device=cpu

Training result:
{'output_dir': '/content/model_lab_smoke_artifacts', 'best_checkpoint': '/content/model_lab_smoke_artifacts/best.pt', 'last_checkpoint': '/content/model_lab_smoke_artifacts/last.pt', 'tokenizer_path': '/content/model_lab_smoke_artifacts/tokenizer.json', 'steps': 30, 'best_val_loss': 2.862246831258138, 'final_train_loss': 2.8625126679738364, 'final_val_loss': 2.862246831258138, 'device': 'cpu'}

SUCCESS: Training and checkpoint creation completed.


In [12]:
import sys
from pathlib import Path

repo = Path("/content/ai-engineering-platform")
sys.path.insert(0, str(repo / "src"))

from ai_platform.machine_learning.language_model.training import (
    load_language_model,
    generate_text,
)

output_dir = Path("/content/model_lab_smoke_artifacts")
checkpoint_path = output_dir / "best.pt"

# إعادة تحميل النموذج المحفوظ
model, tokenizer, checkpoint = load_language_model(
    checkpoint_path,
    device="cpu",
)

print("Checkpoint step:", checkpoint["step"])
print("Vocabulary size:", tokenizer.vocab_size)
print("Model loaded:", model.training is False)

# توليد نص جديد من النموذج المحفوظ
generated = generate_text(
    checkpoint_path,
    prompt="AI engineering",
    max_new_tokens=100,
    temperature=0.8,
    top_k=10,
    device="cpu",
)

print("\nGenerated text:")
print(generated)

assert generated.startswith("AI engineering")
print("\nSUCCESS: Reload and generation completed.")


Checkpoint step: 30
Vocabulary size: 33
Model loaded: True

Generated text:
AI engineering ertaane rntmeranenarartoo aanrritinteesos.orns.,  ltelinrenoago ars cs.menters.iotoooatsxxaioersere

SUCCESS: Reload and generation completed.


In [13]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU unavailable; the current session is using CPU.")

PyTorch: 2.11.0+cpu
CUDA available: False
GPU unavailable; the current session is using CPU.


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

!nvidia-smi

PyTorch: 2.11.0+cu130
CUDA available: True
Mon Oct 19 15:08:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             16W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----

In [2]:
import os
import sys
import subprocess
from pathlib import Path

repo = Path("/content/ai-engineering-platform")

# جلب المشروع أو تحديثه
if not repo.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/F13342/ai-engineering-platform.git",
            str(repo),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", "main"],
        cwd=repo,
        check=True,
    )

os.chdir(repo)

# التأكد من بيئة التدريب
import torch

print("Git commit:", subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], text=True
).strip())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# تثبيت أداة الاختبارات فقط عند الحاجة
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pytest"],
    check=True,
)

# جعل حزمة المشروع متاحة لعملية pytest
env = os.environ.copy()
env["PYTHONPATH"] = str(repo / "src") + os.pathsep + env.get("PYTHONPATH", "")

result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q", "-ra",
        "tests/test_language_model.py",
        "tests/test_language_model_tokenizer.py",
        "tests/test_language_model_training.py",
    ],
    env=env,
    text=True,
    capture_output=True,
)

print("\n--- TEST RESULTS ---")
print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("فشلت بعض الاختبارات؛ سنعالجها قبل التدريب.")

Git commit: 54b97b1
CUDA available: True
GPU: Tesla T4

--- TEST RESULTS ---

no tests ran in 0.00s

ERROR: file or directory not found: tests/test_language_model.py




RuntimeError: فشلت بعض الاختبارات؛ سنعالجها قبل التدريب.

In [3]:
import os
import sys
import subprocess
from pathlib import Path

repo = Path("/content/ai-engineering-platform")

def run(command, **kwargs):
    return subprocess.run(command, check=True, **kwargs)

# جلب أحدث نسخة من GitHub
run(["git", "fetch", "origin", "main"], cwd=repo)

# مزامنة نسخة Colab مع main على GitHub
run(
    ["git", "checkout", "--force", "-B", "main", "origin/main"],
    cwd=repo,
)

os.chdir(repo)

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    text=True,
).strip()

print("Current commit:", commit)
print("Language-model tests exist:",
      (repo / "tests/test_language_model.py").is_file())

assert (repo / "tests/test_language_model.py").is_file(), \
    "Language-model test file is still missing."

# تشغيل الاختبارات بالمسار الصحيح
env = os.environ.copy()
env["PYTHONPATH"] = str(repo / "src") + os.pathsep + env.get("PYTHONPATH", "")

result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q", "-ra",
        "tests/test_language_model.py",
        "tests/test_language_model_tokenizer.py",
        "tests/test_language_model_training.py",
    ],
    cwd=repo,
    env=env,
    text=True,
    capture_output=True,
)

print("\n--- TEST RESULTS ---")
print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Some tests failed; training has not been started.")

print("SUCCESS: All language-model tests passed.")

Current commit: bcd3a6c
Language-model tests exist: True

--- TEST RESULTS ---
.................                                                        [100%]
17 passed in 7.05s

SUCCESS: All language-model tests passed.


In [4]:
import os
import sys
from pathlib import Path
import torch

repo = Path("/content/ai-engineering-platform").resolve()
os.chdir(repo)
sys.path.insert(0, str(repo / "src"))

assert torch.cuda.is_available(), "CUDA is unavailable."
print("GPU:", torch.cuda.get_device_name(0))

from ai_platform.machine_learning.language_model.training import (
    LanguageModelTrainingConfig,
    train_language_model,
    load_language_model,
    generate_text,
)

# بيانات اصطناعية لاختبار خط التدريب، وليست بيانات تدريب نهائية
corpus_text = (
    "AI engineering platform trains its own language model from random initialization. "
    "A tokenizer converts characters into token IDs. A transformer predicts the next "
    "character from previous characters. Training updates parameters and saves checkpoints. "
) * 120

corpus_path = Path("/content/model_lab_gpu_corpus.txt")
corpus_path.write_text(corpus_text, encoding="utf-8")

output_dir = Path("/content/model_lab_gpu_artifacts")

config = LanguageModelTrainingConfig(
    block_size=32,
    n_layers=2,
    n_heads=2,
    n_embd=64,
    dropout=0.1,
    batch_size=16,
    max_steps=30,
    learning_rate=3e-4,
    eval_interval=10,
    eval_batches=3,
    train_fraction=0.9,
    seed=42,
    device="cuda",
)

result = train_language_model(corpus_path, output_dir, config)
print("\nTraining result:")
print(result)

# إعادة تحميل أفضل checkpoint على GPU
model, tokenizer, checkpoint = load_language_model(
    output_dir / "best.pt",
    device="cuda",
)

assert next(model.parameters()).is_cuda
assert (output_dir / "last.pt").is_file()
assert (output_dir / "tokenizer.json").is_file()
assert (output_dir / "metrics.json").is_file()

generated = generate_text(
    output_dir / "best.pt",
    prompt="AI engineering",
    max_new_tokens=40,
    temperature=0.8,
    top_k=10,
    device="cuda",
)

print("\nGenerated text:")
print(generated)
print("\nCheckpoint step:", checkpoint["step"])
print("SUCCESS: GPU training, checkpoint reload, and generation completed.")

GPU: Tesla T4
step 10/30 | train_loss=3.1426 | val_loss=3.1456 | device=cuda
step 20/30 | train_loss=2.9712 | val_loss=2.9854 | device=cuda
step 30/30 | train_loss=2.8208 | val_loss=2.8334 | device=cuda

Training result:
{'output_dir': '/content/model_lab_gpu_artifacts', 'best_checkpoint': '/content/model_lab_gpu_artifacts/best.pt', 'last_checkpoint': '/content/model_lab_gpu_artifacts/last.pt', 'tokenizer_path': '/content/model_lab_gpu_artifacts/tokenizer.json', 'steps': 30, 'best_val_loss': 2.8333882490793862, 'final_train_loss': 2.8208460013071694, 'final_val_loss': 2.8333882490793862, 'device': 'cuda'}

Generated text:
AI engineering aatinsios traneormatoateniings toarrspn

Checkpoint step: 30
SUCCESS: GPU training, checkpoint reload, and generation completed.


In [5]:
from google.colab import files

uploaded = files.upload()

Saving pg11.txt to pg11.txt


In [6]:
from pathlib import Path

path = Path("/content/pg11.txt")
assert path.is_file(), "لم يتم العثور على الملف pg11.txt"

text = path.read_text(encoding="utf-8", errors="replace")
lines = text.splitlines()

print("File:", path.name)
print("Characters:", len(text))
print("Lines:", len(lines))

print("\n--- Beginning of file ---")
print("\n".join(lines[:15]))

print("\n--- Gutenberg markers ---")
for i, line in enumerate(lines):
    upper = line.upper()
    if "START OF THE PROJECT GUTENBERG" in upper:
        print(f"Start marker at line {i + 1}: {line.strip()}")
    if "END OF THE PROJECT GUTENBERG" in upper:
        print(f"End marker at line {i + 1}: {line.strip()}")

print("\n--- End of file ---")
print("\n".join(lines[-15:]))

AssertionError: لم يتم العثور على الملف pg11.txt

In [7]:
from pathlib import Path
import shutil

matches = list(Path("/content").rglob("pg11.txt"))

print("Files found:", [str(p) for p in matches])

if not matches:
    raise FileNotFoundError(
        "لم يُعثر على pg11.txt داخل /content. أعد رفع الملف باستخدام files.upload()."
    )

source = matches[0]
destination = Path("/content/pg11.txt")

if source.resolve() != destination.resolve():
    shutil.copy2(source, destination)

print("SUCCESS: File is ready at", destination)
print("Size:", destination.stat().st_size, "bytes")

Files found: ['/content/ai-engineering-platform/pg11.txt']
SUCCESS: File is ready at /content/pg11.txt
Size: 174311 bytes


In [8]:
from pathlib import Path

path = Path("/content/pg11.txt")
text = path.read_text(encoding="utf-8", errors="replace")
lines = text.splitlines()

print("File:", path.name)
print("Characters:", len(text))
print("Lines:", len(lines))

print("\n--- Beginning of file ---")
print("\n".join(lines[:12]))

print("\n--- Gutenberg markers ---")
for i, line in enumerate(lines):
    upper = line.upper()
    if "START OF THE PROJECT GUTENBERG" in upper:
        print(f"Start marker — line {i + 1}: {line.strip()}")
    if "END OF THE PROJECT GUTENBERG" in upper:
        print(f"End marker — line {i + 1}: {line.strip()}")

print("\n--- End of file ---")
print("\n".join(lines[-12:]))

File: pg11.txt
Characters: 163950
Lines: 3762

--- Beginning of file ---
The Project Gutenberg eBook of Alice's Adventures in Wonderland
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

Title: Alice's Adventures in Wonderland


--- Gutenberg markers ---
Start marker — line 27: *** START OF THE PROJECT GUTENBERG EBOOK ALICE'S ADVENTURES IN WONDERLAND ***
End marker — line 3412: *** END OF THE PROJECT GUTENBERG EBOOK ALICE'S ADVENTURES IN WONDERLAND ***

--- End of file ---
necessarily keep eBooks in compliance with any particular paper
edition.

Most people start at our website which has the ma

In [9]:
from pathlib import Path

path = Path("/content/pg11.txt")
text = path.read_text(encoding="utf-8", errors="replace")
lines = text.splitlines()

print("File:", path.name)
print("Characters:", len(text))
print("Lines:", len(lines))

print("\n--- Beginning of file ---")
print("\n".join(lines[:12]))

print("\n--- Gutenberg markers ---")
for i, line in enumerate(lines):
    upper = line.upper()
    if "START OF THE PROJECT GUTENBERG" in upper:
        print(f"Start marker — line {i + 1}: {line.strip()}")
    if "END OF THE PROJECT GUTENBERG" in upper:
        print(f"End marker — line {i + 1}: {line.strip()}")

print("\n--- End of file ---")
print("\n".join(lines[-12:]))

File: pg11.txt
Characters: 163950
Lines: 3762

--- Beginning of file ---
The Project Gutenberg eBook of Alice's Adventures in Wonderland
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

Title: Alice's Adventures in Wonderland


--- Gutenberg markers ---
Start marker — line 27: *** START OF THE PROJECT GUTENBERG EBOOK ALICE'S ADVENTURES IN WONDERLAND ***
End marker — line 3412: *** END OF THE PROJECT GUTENBERG EBOOK ALICE'S ADVENTURES IN WONDERLAND ***

--- End of file ---
necessarily keep eBooks in compliance with any particular paper
edition.

Most people start at our website which has the ma

In [10]:
from pathlib import Path
import hashlib
import json
import re
import unicodedata
from datetime import datetime, timezone

# الملف الأصلي الذي رفعته
source_path = Path("/content/pg11.txt")
assert source_path.is_file(), "لم يُعثر على ملف pg11.txt"

raw_text = source_path.read_text(encoding="utf-8", errors="replace")

# تحديد علامتي البداية والنهاية
start_match = re.search(
    r"\*\*\*\s*START OF THE PROJECT GUTENBERG EBOOK[^\n]*\*\*\*",
    raw_text,
    flags=re.IGNORECASE,
)
end_match = re.search(
    r"\*\*\*\s*END OF THE PROJECT GUTENBERG EBOOK[^\n]*\*\*\*",
    raw_text,
    flags=re.IGNORECASE,
)

assert start_match, "لم نعثر على علامة بداية الكتاب"
assert end_match, "لم نعثر على علامة نهاية الكتاب"
assert start_match.end() < end_match.start(), "ترتيب العلامات غير صحيح"

# استخراج متن الكتاب فقط
clean_text = raw_text[start_match.end():end_match.start()]
clean_text = unicodedata.normalize("NFC", clean_text)
clean_text = clean_text.replace("\r\n", "\n").replace("\r", "\n")

# تنظيف المسافات الزائدة دون حذف فواصل الفقرات
clean_text = "\n".join(line.rstrip() for line in clean_text.splitlines())
clean_text = re.sub(r"\n{3,}", "\n\n", clean_text).strip() + "\n"

assert len(clean_text) > 10_000, "النص المستخرج أقصر من المتوقع"
assert "START OF THE PROJECT GUTENBERG EBOOK" not in clean_text
assert "END OF THE PROJECT GUTENBERG EBOOK" not in clean_text

# حفظ البيانات خارج مستودع GitHub
output_dir = Path("/content/model_lab_data")
output_dir.mkdir(parents=True, exist_ok=True)

corpus_path = output_dir / "alice_adventures_in_wonderland.txt"
metadata_path = output_dir / "metadata.json"

corpus_path.write_text(clean_text, encoding="utf-8")

metadata = {
    "title": "Alice's Adventures in Wonderland",
    "author": "Lewis Carroll",
    "source_organization": "Project Gutenberg",
    "source_page": "https://www.gutenberg.org/ebooks/11",
    "source_file": "pg11.txt",
    "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
    "cleaning_method": (
        "Extracted text between Project Gutenberg start/end markers; "
        "normalized Unicode NFC and line endings; trimmed trailing whitespace."
    ),
    "character_count": len(clean_text),
    "line_count": len(clean_text.splitlines()),
    "word_count_approx": len(clean_text.split()),
    "sha256": hashlib.sha256(clean_text.encode("utf-8")).hexdigest(),
    "rights_note": (
        "Project Gutenberg describes this eBook as public domain in the United "
        "States. Users outside the United States must check applicable local law. "
        "Review relevant terms and rights before commercial redistribution or use."
    ),
}

metadata_path.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print("SUCCESS: Text cleaned and metadata recorded.")
print("Corpus path:", corpus_path)
print("Metadata path:", metadata_path)
print("Characters:", metadata["character_count"])
print("Approximate words:", metadata["word_count_approx"])
print("Lines:", metadata["line_count"])
print("SHA-256:", metadata["sha256"])

print("\n--- Clean text preview ---")
print(clean_text[:700])

print("\n--- End preview ---")
print(clean_text[-400:])

print("\n--- Rights note ---")
print(metadata["rights_note"])

SUCCESS: Text cleaned and metadata recorded.
Corpus path: /content/model_lab_data/alice_adventures_in_wonderland.txt
Metadata path: /content/model_lab_data/metadata.json
Characters: 144530
Approximate words: 26525
Lines: 3310
SHA-256: a4c81ef23eb99f8b14be2474be0410b708cc99293ce6d88cf6799335926639b9

--- Clean text preview ---
[Illustration]

Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The Mock Turtle’s Story
 CHAPTER X.     The Lobster Quadrille
 CHAPTER XI.    Who Stole the Tarts?
 CHAPTER XII.   Alice’s Evidence

CHAPTER I.
Down the Rabbit-Hole

Alice was beginning to get very tired of sitting by her siste

In [11]:
import os
import sys
import json
import shutil
from pathlib import Path

import torch

repo = Path("/content/ai-engineering-platform").resolve()
sys.path.insert(0, str(repo / "src"))

corpus_path = Path(
    "/content/model_lab_data/alice_adventures_in_wonderland.txt"
)
metadata_path = Path("/content/model_lab_data/metadata.json")
output_dir = Path("/content/alice_model_experiment_v1")

# التأكد من جاهزية البيئة والبيانات
assert torch.cuda.is_available(), "GPU غير متاح في جلسة Colab الحالية."
assert corpus_path.is_file(), "ملف الرواية غير موجود."
assert metadata_path.is_file(), "ملف بيانات المصدر والترخيص غير موجود."

metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

print("GPU:", torch.cuda.get_device_name(0))
print("Corpus characters:", len(
    corpus_path.read_text(encoding="utf-8")
))
print("Source:", metadata["source_page"])
print("Data SHA-256:", metadata["sha256"])

from ai_platform.machine_learning.language_model.training import (
    LanguageModelTrainingConfig,
    train_language_model,
)

# تجربة تدريب أولية من الصفر على النص الحقيقي
config = LanguageModelTrainingConfig(
    block_size=128,
    n_layers=4,
    n_heads=4,
    n_embd=128,
    dropout=0.1,
    batch_size=32,
    max_steps=1000,
    learning_rate=3e-4,
    weight_decay=0.01,
    eval_interval=100,
    eval_batches=10,
    train_fraction=0.9,
    grad_clip=1.0,
    seed=42,
    device="cuda",
)

result = train_language_model(
    corpus_path=corpus_path,
    output_dir=output_dir,
    config=config,
)

# الاحتفاظ بسجل مصدر البيانات مع ملفات التجربة
shutil.copy2(metadata_path, output_dir / "dataset_metadata.json")

print("\n" + "=" * 45)
print("TRAINING SUMMARY")
print("=" * 45)
print(json.dumps(result, indent=2, ensure_ascii=False))

assert (output_dir / "best.pt").is_file()
assert (output_dir / "last.pt").is_file()
assert (output_dir / "tokenizer.json").is_file()
assert (output_dir / "metrics.json").is_file()
assert (output_dir / "dataset_metadata.json").is_file()

print("\nSUCCESS: Real-text training completed.")
print("Artifacts directory:", output_dir)

GPU: Tesla T4
Corpus characters: 144530
Source: https://www.gutenberg.org/ebooks/11
Data SHA-256: a4c81ef23eb99f8b14be2474be0410b708cc99293ce6d88cf6799335926639b9
step 100/1000 | train_loss=2.5722 | val_loss=2.5511 | device=cuda
step 200/1000 | train_loss=2.4524 | val_loss=2.4612 | device=cuda
step 300/1000 | train_loss=2.3671 | val_loss=2.3835 | device=cuda
step 400/1000 | train_loss=2.3386 | val_loss=2.3453 | device=cuda
step 500/1000 | train_loss=2.2253 | val_loss=2.2350 | device=cuda
step 600/1000 | train_loss=2.1279 | val_loss=2.1637 | device=cuda
step 700/1000 | train_loss=2.0059 | val_loss=2.0508 | device=cuda
step 800/1000 | train_loss=1.9491 | val_loss=1.9891 | device=cuda
step 900/1000 | train_loss=1.8569 | val_loss=1.9148 | device=cuda
step 1000/1000 | train_loss=1.7913 | val_loss=1.8374 | device=cuda

TRAINING SUMMARY
{
  "output_dir": "/content/alice_model_experiment_v1",
  "best_checkpoint": "/content/alice_model_experiment_v1/best.pt",
  "last_checkpoint": "/content/alic

In [12]:
import json
import zipfile
from pathlib import Path
from IPython.display import display
from google.colab import files

from ai_platform.machine_learning.language_model.training import (
    load_language_model,
    generate_text,
)

output_dir = Path("/content/alice_model_experiment_v1")
checkpoint = output_dir / "best.pt"

# تحميل النموذج المحفوظ للتحقق من إمكانية استخدامه
model, tokenizer, checkpoint_info = load_language_model(
    checkpoint,
    device="cuda",
)

print("Checkpoint step:", checkpoint_info["step"])
print("Vocabulary size:", tokenizer.vocab_size)
print("Device:", next(model.parameters()).device)

# توليد عينات من مطالبات مختلفة
prompts = [
    "Alice was beginning",
    "Down the Rabbit-Hole",
    "The Queen",
]

for prompt in prompts:
    print("\n" + "=" * 60)
    print("PROMPT:", prompt)
    print("=" * 60)

    sample = generate_text(
        checkpoint,
        prompt=prompt,
        max_new_tokens=180,
        temperature=0.8,
        top_k=20,
        device="cuda",
    )
    print(sample)

# التأكد من وجود الملفات الأساسية
required_files = [
    "best.pt",
    "last.pt",
    "tokenizer.json",
    "metrics.json",
    "dataset_metadata.json",
]

for name in required_files:
    path = output_dir / name
    assert path.is_file(), f"Missing artifact: {name}"
    print(f"Verified: {name} ({path.stat().st_size:,} bytes)")

# إنشاء نسخة احتياطية قابلة للتنزيل
archive_path = Path("/content/alice_model_experiment_v1.zip")

with zipfile.ZipFile(
    archive_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for name in required_files:
        archive.write(output_dir / name, arcname=name)

print("\nSUCCESS: Checkpoint reload, generation, and archive creation completed.")
print("Archive size:", f"{archive_path.stat().st_size:,}", "bytes")

files.download(str(archive_path))

Checkpoint step: 1000
Vocabulary size: 79
Device: cuda:0

PROMPT: Alice was beginning
Alice was beginning,in and and beatter had
doners: “Wel that al to the ad wo drite, I af you’ I’very was _to the Mat Tuch me
was and what pof of replden the by wit bear a the Moch Rabatll Holeear
 do

PROMPT: Down the Rabbit-Hole
Down the Rabbit-Holed,
the Mave Finter the she pupped a
the mallity the a his wered and fittler the she seent, and bout leave
wats ond ind a as the por she that bered ooused an manlle. Her is wen to l

PROMPT: The Queen
The Queent, and the Careply: “Is’s she be ever, and
fery to go, and crould wond the testin.”

“What thime you werdely of wouldnt boon to a a mare worte willk thing a
houscked voulde the wa 
Verified: best.pt (3,296,025 bytes)
Verified: last.pt (3,296,025 bytes)
Verified: tokenizer.json (816 bytes)
Verified: metrics.json (987 bytes)
Verified: dataset_metadata.json (821 bytes)

SUCCESS: Checkpoint reload, generation, and archive creation completed.
Archive size:

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
import os
import sys
import subprocess
from pathlib import Path

repo = Path("/content/ai-engineering-platform").resolve()

# تحديث المشروع إلى آخر نسخة على GitHub
subprocess.run(["git", "fetch", "origin", "main"], cwd=repo, check=True)
subprocess.run(
    ["git", "checkout", "--force", "-B", "main", "origin/main"],
    cwd=repo,
    check=True,
)

os.chdir(repo)

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], text=True
).strip()

import torch

print("Commit:", commit)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# تجهيز الاختبارات
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pytest"],
    check=True,
)

env = os.environ.copy()
env["PYTHONPATH"] = str(repo / "src") + os.pathsep + env.get("PYTHONPATH", "")

result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q", "-ra",
        "tests/test_language_model.py",
        "tests/test_language_model_tokenizer.py",
        "tests/test_language_model_bpe_tokenizer.py",
        "tests/test_language_model_training.py",
    ],
    cwd=repo,
    env=env,
    text=True,
    capture_output=True,
)

print("\n--- TEST RESULTS ---")
print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Some language-model tests failed; v2 training has not started.")

print("SUCCESS: Language-model tests passed.")

Commit: 30d03e1
CUDA available: True
GPU: Tesla T4

--- TEST RESULTS ---
..............................                                           [100%]
30 passed in 2.97s

SUCCESS: Language-model tests passed.


In [14]:
import os
import sys
import json
import shutil
from pathlib import Path

import torch

repo = Path("/content/ai-engineering-platform").resolve()
os.chdir(repo)
sys.path.insert(0, str(repo / "src"))

corpus_path = Path(
    "/content/model_lab_data/alice_adventures_in_wonderland.txt"
)
metadata_path = Path("/content/model_lab_data/metadata.json")
output_dir = Path("/content/alice_model_experiment_v2_bpe")

assert torch.cuda.is_available(), "Tesla T4 GPU is not available."
assert corpus_path.is_file(), "Cleaned Alice corpus is missing."
assert metadata_path.is_file(), "Dataset metadata is missing."

print("GPU:", torch.cuda.get_device_name(0))
print("Dataset characters:", len(corpus_path.read_text(encoding="utf-8")))

from ai_platform.machine_learning.language_model.training import (
    LanguageModelTrainingConfig,
    train_language_model,
)

config = LanguageModelTrainingConfig(
    block_size=128,
    n_layers=4,
    n_heads=4,
    n_embd=128,
    dropout=0.1,
    batch_size=32,
    max_steps=1000,
    learning_rate=3e-4,
    weight_decay=0.01,
    eval_interval=100,
    eval_batches=10,
    train_fraction=0.9,
    grad_clip=1.0,
    seed=42,
    tokenizer_type="bpe",
    tokenizer_vocab_size=512,
    tokenizer_min_frequency=2,
    device="cuda",
)

result = train_language_model(
    corpus_path=corpus_path,
    output_dir=output_dir,
    config=config,
)

shutil.copy2(metadata_path, output_dir / "dataset_metadata.json")

# Record the experiment configuration and results.
experiment = {
    "experiment": "Alice Model v2 — BPE",
    "tokenizer": "BPE trained from scratch",
    "dataset": "Alice's Adventures in Wonderland",
    "training_config": config.__dict__,
    "results": result,
}

(output_dir / "experiment.json").write_text(
    json.dumps(experiment, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print("\n--- V2 TRAINING SUMMARY ---")
print(json.dumps(result, ensure_ascii=False, indent=2))
print("Experiment directory:", output_dir)

for name in [
    "best.pt",
    "last.pt",
    "tokenizer.json",
    "metrics.json",
    "dataset_metadata.json",
    "experiment.json",
]:
    artifact = output_dir / name
    assert artifact.is_file(), f"Missing artifact: {name}"
    print(f"Verified {name}: {artifact.stat().st_size:,} bytes")

print("\nSUCCESS: Alice Model v2 training completed.")


GPU: Tesla T4
Dataset characters: 144530


TypeError: LanguageModelTrainingConfig.__init__() got an unexpected keyword argument 'tokenizer_type'

In [15]:
import os
import sys
import importlib
from pathlib import Path

repo = Path("/content/ai-engineering-platform").resolve()
os.chdir(repo)

src = str(repo / "src")
if src in sys.path:
    sys.path.remove(src)
sys.path.insert(0, src)

importlib.invalidate_caches()

import ai_platform.machine_learning.language_model.training as training_module
training_module = importlib.reload(training_module)

LanguageModelTrainingConfig = training_module.LanguageModelTrainingConfig
train_language_model = training_module.train_language_model

print("Loaded module:", training_module.__file__)
print("Supports tokenizer_type:",
      "tokenizer_type" in LanguageModelTrainingConfig.__dataclass_fields__)
print("Supports tokenizer_vocab_size:",
      "tokenizer_vocab_size" in LanguageModelTrainingConfig.__dataclass_fields__)

assert "tokenizer_type" in LanguageModelTrainingConfig.__dataclass_fields__, (
    "The active module is still outdated. Do not start training."
)

print("SUCCESS: Latest training module is loaded.")

Loaded module: /content/ai-engineering-platform/src/ai_platform/machine_learning/language_model/training.py
Supports tokenizer_type: True
Supports tokenizer_vocab_size: True
SUCCESS: Latest training module is loaded.


In [16]:
import os
import sys
import json
import shutil
import importlib
from pathlib import Path

import torch

# 1. تجهيز المشروع
repo = Path("/content/ai-engineering-platform").resolve()
os.chdir(repo)

src_path = str(repo / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# إعادة تحميل نسخة التدريب الحديثة
import ai_platform.machine_learning.language_model.training as training_module

importlib.invalidate_caches()
training_module = importlib.reload(training_module)

# 2. التحقق من البيئة والبيانات
assert torch.cuda.is_available(), "فعّل Tesla T4 أولًا."
assert "tokenizer_type" in training_module.LanguageModelTrainingConfig.__dataclass_fields__, (
    "نسخة التدريب قديمة؛ حدّث المشروع من GitHub."
)

corpus_path = Path(
    "/content/model_lab_data/alice_adventures_in_wonderland.txt"
)
metadata_path = Path("/content/model_lab_data/metadata.json")

assert corpus_path.is_file(), "ملف الرواية المنظّف غير موجود."
assert metadata_path.is_file(), "ملف بيانات المصدر والترخيص غير موجود."

print("GPU:", torch.cuda.get_device_name(0))
print("Commit:", subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=repo,
    text=True,
).strip() if (repo / ".git").exists() else "unknown")
print("Corpus characters:", len(corpus_path.read_text(encoding="utf-8")))

# 3. إعداد Alice Model v2
output_dir = Path("/content/alice_model_experiment_v2_bpe")

config = training_module.LanguageModelTrainingConfig(
    block_size=128,
    n_layers=4,
    n_heads=4,
    n_embd=128,
    dropout=0.1,
    batch_size=32,
    max_steps=1000,
    learning_rate=3e-4,
    weight_decay=0.01,
    eval_interval=100,
    eval_batches=10,
    train_fraction=0.9,
    grad_clip=1.0,
    seed=42,
    tokenizer_type="bpe",
    tokenizer_vocab_size=512,
    tokenizer_min_frequency=2,
    device="cuda",
)

# 4. التدريب من أوزان عشوائية
result = training_module.train_language_model(
    corpus_path=corpus_path,
    output_dir=output_dir,
    config=config,
)

# 5. حفظ سجل مصدر البيانات والتجربة
shutil.copy2(metadata_path, output_dir / "dataset_metadata.json")

experiment = {
    "experiment": "Alice Model v2 - BPE",
    "initialization": "random",
    "tokenizer": "BPE trained from scratch on the training split",
    "dataset": "Alice's Adventures in Wonderland",
    "training_config": config.__dict__,
    "results": result,
}

(output_dir / "experiment.json").write_text(
    json.dumps(experiment, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

# 6. التحقق من الملفات الناتجة
required_files = [
    "best.pt",
    "last.pt",
    "tokenizer.json",
    "metrics.json",
    "dataset_metadata.json",
    "experiment.json",
]

print("\n--- ALICE MODEL V2 SUMMARY ---")
print(json.dumps(result, ensure_ascii=False, indent=2))

for name in required_files:
    path = output_dir / name
    assert path.is_file(), f"ملف مطلوب مفقود: {name}"
    print(f"Verified {name}: {path.stat().st_size:,} bytes")

print("\nSUCCESS: Alice Model v2 training completed.")
print("Artifacts:", output_dir)

GPU: Tesla T4
Commit: 30d03e1
Corpus characters: 144530
step 100/1000 | train_loss=3.7820 | val_loss=3.8243 | device=cuda
step 200/1000 | train_loss=3.4700 | val_loss=3.5265 | device=cuda
step 300/1000 | train_loss=3.3248 | val_loss=3.3934 | device=cuda
step 400/1000 | train_loss=3.1784 | val_loss=3.3064 | device=cuda
step 500/1000 | train_loss=3.0629 | val_loss=3.1837 | device=cuda
step 600/1000 | train_loss=2.9653 | val_loss=3.1379 | device=cuda
step 700/1000 | train_loss=2.8528 | val_loss=3.0542 | device=cuda
step 800/1000 | train_loss=2.7388 | val_loss=2.9741 | device=cuda
step 900/1000 | train_loss=2.6380 | val_loss=2.9063 | device=cuda
step 1000/1000 | train_loss=2.5352 | val_loss=2.8485 | device=cuda

--- ALICE MODEL V2 SUMMARY ---
{
  "output_dir": "/content/alice_model_experiment_v2_bpe",
  "best_checkpoint": "/content/alice_model_experiment_v2_bpe/best.pt",
  "last_checkpoint": "/content/alice_model_experiment_v2_bpe/last.pt",
  "tokenizer_path": "/content/alice_model_experim

In [17]:
import sys
import json
import zipfile
from pathlib import Path

repo = Path("/content/ai-engineering-platform").resolve()
sys.path.insert(0, str(repo / "src"))

from ai_platform.machine_learning.language_model.training import (
    load_language_model,
    generate_text,
)

v1_dir = Path("/content/alice_model_experiment_v1")
v2_dir = Path("/content/alice_model_experiment_v2_bpe")

v1_checkpoint = v1_dir / "best.pt"
v2_checkpoint = v2_dir / "best.pt"

assert v2_checkpoint.is_file(), "ملف أفضل أوزان v2 غير موجود."
assert (v2_dir / "tokenizer.json").is_file(), "مفردات BPE غير موجودة."

# تحميل v2 والتحقق من checkpoint
model, tokenizer, checkpoint = load_language_model(
    v2_checkpoint,
    device="cuda",
)

print("V2 checkpoint step:", checkpoint["step"])
print("V2 vocabulary size:", tokenizer.vocab_size)
print("V2 device:", next(model.parameters()).device)

prompts = [
    "Alice was beginning",
    "Down the Rabbit-Hole",
    "The Queen",
]

samples = {}

for prompt in prompts:
    print("\n" + "=" * 60)
    print("PROMPT:", prompt)

    sample_v2 = generate_text(
        v2_checkpoint,
        prompt=prompt,
        max_new_tokens=120,
        temperature=0.8,
        top_k=20,
        device="cuda",
    )

    samples[prompt] = {"v2_bpe": sample_v2}
    print("\nV2 — BPE:")
    print(sample_v2)

    # مقارنة نوعية اختيارية إذا كانت أوزان v1 لا تزال موجودة
    if (v1_dir / "best.pt").is_file():
        sample_v1 = generate_text(
            v1_dir / "best.pt",
            prompt=prompt,
            max_new_tokens=120,
            temperature=0.8,
            top_k=20,
            device="cuda",
        )
        samples[prompt]["v1_character"] = sample_v1
        print("\nV1 — Character:")
        print(sample_v1)

# حفظ عينات التوليد ضمن مجلد v2
samples_path = v2_dir / "generation_samples.json"
samples_path.write_text(
    json.dumps(samples, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

# التأكد من وجود الملفات المطلوبة قبل إنشاء النسخة الاحتياطية
required_files = [
    "best.pt",
    "last.pt",
    "tokenizer.json",
    "metrics.json",
    "dataset_metadata.json",
    "experiment.json",
    "generation_samples.json",
]

for name in required_files:
    assert (v2_dir / name).is_file(), f"ملف مفقود: {name}"

# إنشاء أرشيف مستقل للإصدار الثاني
archive_path = Path("/content/alice_model_experiment_v2_bpe.zip")

with zipfile.ZipFile(
    archive_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for name in required_files:
        archive.write(v2_dir / name, arcname=name)

print("\nSUCCESS: V2 reload and generation completed.")
print("Backup archive:", archive_path)
print("Archive size:", f"{archive_path.stat().st_size:,}", "bytes")

from google.colab import files
files.download(str(archive_path))

V2 checkpoint step: 1000
V2 vocabulary size: 512
V2 device: cuda:0

PROMPT: Alice was beginning

V2 — BPE:
Alice was beginning, and the Queen had fanken indealon and the
drather of crocried Alice to that she went on on on on in up to the table, and was a pair
gards to the little s, and was the little fighing cried her the skneely treen the p

V1 — Character:
Alice was beginning farly. Alice the by wown as s lemet
to thightle foldeal to downden dor with I the she had rown
the malling the a had we

PROMPT: Down the Rabbit-Hole

V2 — BPE:
Down the Rabbit-Hole the was cross.

“And I a ft the shall fives, with In like alled at the cried tried
of it, that it feetly it could not to find to to the broven,” said, “so I I
should to to be a brabbit s her ways and to

V1 — Character:
Down the Rabbit-Hole thate creme and of its, and
the was had befor thought a whas the wond houldes in.

“Now that beer bing and theise itt s

PROMPT: The Queen

V2 — BPE:
The Queen’s book them a little childreamong fo

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
import os
import sys
import json
import shutil
import importlib
from pathlib import Path

import torch

repo = Path("/content/ai-engineering-platform").resolve()
os.chdir(repo)
sys.path.insert(0, str(repo / "src"))

import ai_platform.machine_learning.language_model.training as training_module
importlib.invalidate_caches()
training_module = importlib.reload(training_module)

assert torch.cuda.is_available(), "Tesla T4 GPU is not available."

corpus_path = Path(
    "/content/model_lab_data/alice_adventures_in_wonderland.txt"
)
metadata_path = Path("/content/model_lab_data/metadata.json")
output_dir = Path("/content/alice_model_v1_character_clean")

assert corpus_path.is_file()
assert metadata_path.is_file()

print("GPU:", torch.cuda.get_device_name(0))
print("Baseline: Character tokenizer with leakage-free split")

config = training_module.LanguageModelTrainingConfig(
    block_size=128,
    n_layers=4,
    n_heads=4,
    n_embd=128,
    dropout=0.1,
    batch_size=32,
    max_steps=1000,
    learning_rate=3e-4,
    weight_decay=0.01,
    eval_interval=100,
    eval_batches=10,
    train_fraction=0.9,
    grad_clip=1.0,
    seed=42,
    tokenizer_type="character",
    device="cuda",
)

result = training_module.train_language_model(
    corpus_path=corpus_path,
    output_dir=output_dir,
    config=config,
)

shutil.copy2(metadata_path, output_dir / "dataset_metadata.json")

experiment = {
    "experiment": "Alice Model — Character baseline with corrected split",
    "initialization": "random",
    "tokenizer": "character",
    "training_config": config.__dict__,
    "results": result,
}

(output_dir / "experiment.json").write_text(
    json.dumps(experiment, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print("\n--- CHARACTER BASELINE RESULTS ---")
print(json.dumps(result, ensure_ascii=False, indent=2))
print("\nSUCCESS: Corrected character baseline training completed.")

GPU: Tesla T4
Baseline: Character tokenizer with leakage-free split
step 100/1000 | train_loss=2.5722 | val_loss=2.5511 | device=cuda
step 200/1000 | train_loss=2.4524 | val_loss=2.4612 | device=cuda
step 300/1000 | train_loss=2.3671 | val_loss=2.3835 | device=cuda
step 400/1000 | train_loss=2.3386 | val_loss=2.3453 | device=cuda
step 500/1000 | train_loss=2.2253 | val_loss=2.2350 | device=cuda
step 600/1000 | train_loss=2.1279 | val_loss=2.1637 | device=cuda
step 700/1000 | train_loss=2.0059 | val_loss=2.0508 | device=cuda
step 800/1000 | train_loss=1.9491 | val_loss=1.9891 | device=cuda
step 900/1000 | train_loss=1.8569 | val_loss=1.9148 | device=cuda
step 1000/1000 | train_loss=1.7913 | val_loss=1.8374 | device=cuda

--- CHARACTER BASELINE RESULTS ---
{
  "output_dir": "/content/alice_model_v1_character_clean",
  "best_checkpoint": "/content/alice_model_v1_character_clean/best.pt",
  "last_checkpoint": "/content/alice_model_v1_character_clean/last.pt",
  "tokenizer_path": "/content/

In [19]:
import os
import sys
import json
import math
from pathlib import Path

import torch
import torch.nn.functional as F

repo = Path("/content/ai-engineering-platform").resolve()
os.chdir(repo)
sys.path.insert(0, str(repo / "src"))

import ai_platform.machine_learning.language_model.training as training_module
import importlib
training_module = importlib.reload(training_module)

load_language_model = training_module.load_language_model
generate_text = training_module.generate_text

assert torch.cuda.is_available(), "Tesla T4 GPU is not available."

corpus_path = Path(
    "/content/model_lab_data/alice_adventures_in_wonderland.txt"
)
text = corpus_path.read_text(encoding="utf-8")
text = text.replace("\r\n", "\n").replace("\r", "\n")

# نفس تقسيم التدريب والتحقق لكلا النموذجين
split_at = int(len(text) * 0.9)
val_text = text[split_at:]

models = {
    "Character baseline": Path(
        "/content/alice_model_v1_character_clean/best.pt"
    ),
    "BPE v2": Path(
        "/content/alice_model_experiment_v2_bpe/best.pt"
    ),
}

def evaluate_bpc(checkpoint_path, validation_text):
    model, tokenizer, checkpoint = load_language_model(
        checkpoint_path, device="cuda"
    )
    model.eval()
    device = next(model.parameters()).device
    token_ids = tokenizer.encode(validation_text)

    ids = torch.tensor(token_ids, dtype=torch.long, device=device)
    block_size = checkpoint["model_config"]["block_size"]

    total_nll = 0.0
    target_count = 0

    # تقييم تسلسلي كامل من دون تداخل أهداف المقاطع.
    with torch.no_grad():
        for start in range(0, len(ids) - 1, block_size):
            end = min(start + block_size, len(ids) - 1)

            x = ids[start:end].unsqueeze(0)
            y = ids[start + 1:end + 1].unsqueeze(0)

            logits, _ = model(x)
            loss_sum = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                y.reshape(-1),
                reduction="sum",
            )

            total_nll += loss_sum.item()
            target_count += y.numel()

    bits_per_character = (
        total_nll / math.log(2) / len(validation_text)
    )

    return {
        "checkpoint_step": checkpoint["step"],
        "vocabulary_size": tokenizer.vocab_size,
        "validation_characters": len(validation_text),
        "validation_tokens": len(token_ids),
        "tokens_per_character": len(token_ids) / len(validation_text),
        "average_nll_per_token_nats": total_nll / target_count,
        "bits_per_character": bits_per_character,
    }

results = {}

for name, checkpoint_path in models.items():
    assert checkpoint_path.is_file(), f"Checkpoint missing: {checkpoint_path}"
    results[name] = evaluate_bpc(checkpoint_path, val_text)

print("=== HELD-OUT EVALUATION ===")
print(json.dumps(results, ensure_ascii=False, indent=2))

char_bpc = results["Character baseline"]["bits_per_character"]
bpe_bpc = results["BPE v2"]["bits_per_character"]

print("\n=== COMPARISON ===")
print(f"Character BPC: {char_bpc:.4f}")
print(f"BPE BPC:       {bpe_bpc:.4f}")

if bpe_bpc < char_bpc:
    print("BPE has lower measured bits per character on this split.")
elif bpe_bpc > char_bpc:
    print("Character baseline has lower measured bits per character on this split.")
else:
    print("Both have the same measured bits per character.")

# مقارنة التوليد من المطالبات نفسها
prompts = [
    "Alice was beginning",
    "Down the Rabbit-Hole",
    "The Queen",
]

samples = {}

for prompt in prompts:
    samples[prompt] = {}
    for name, checkpoint_path in models.items():
        sample = generate_text(
            checkpoint_path,
            prompt=prompt,
            max_new_tokens=120,
            temperature=0.8,
            top_k=20,
            device="cuda",
        )
        samples[prompt][name] = sample
        print(f"\n--- {name} | {prompt} ---")
        print(sample)

report = {
    "validation_method": (
        "Sequential full validation split; non-overlapping target blocks; "
        "BPC normalized by raw validation character count."
    ),
    "results": results,
    "generation_samples": samples,
}

report_path = Path("/content/alice_model_comparison.json")
report_path.write_text(
    json.dumps(report, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print("\nSUCCESS: Comparison report saved to", report_path)

=== HELD-OUT EVALUATION ===
{
  "Character baseline": {
    "checkpoint_step": 1000,
    "vocabulary_size": 79,
    "validation_characters": 14453,
    "validation_tokens": 14453,
    "tokens_per_character": 1.0,
    "average_nll_per_token_nats": 1.8591357805773125,
    "bits_per_character": 2.681980392484315
  },
  "BPE v2": {
    "checkpoint_step": 1000,
    "vocabulary_size": 512,
    "validation_characters": 14453,
    "validation_tokens": 7758,
    "tokens_per_character": 0.5367743721026776,
    "average_nll_per_token_nats": 2.8622045719824887,
    "bits_per_character": 2.216210452429439
  }
}

=== COMPARISON ===
Character BPC: 2.6820
BPE BPC:       2.2162
BPE has lower measured bits per character on this split.

--- Character baseline | Alice was beginning ---
Alice was beginning,in and and beatter had
doners: “Wel that al to the ad wo drite, I af you’ I’very was _to the Mat Tuch me
was and what p

--- BPE v2 | Alice was beginning ---
Alice was beginning that for it.

“You was _ 

In [20]:
import json
import zipfile
from pathlib import Path
from google.colab import files

v1_dir = Path("/content/alice_model_v1_character_clean")
v2_dir = Path("/content/alice_model_experiment_v2_bpe")
comparison_path = Path("/content/alice_model_comparison.json")

assert v1_dir.is_dir(), "Character baseline artifacts are missing."
assert v2_dir.is_dir(), "BPE v2 artifacts are missing."
assert comparison_path.is_file(), "Comparison report is missing."

report = json.loads(comparison_path.read_text(encoding="utf-8"))

char_bpc = report["results"]["Character baseline"]["bits_per_character"]
bpe_bpc = report["results"]["BPE v2"]["bits_per_character"]

summary = {
    "comparison": "Alice Model v1 Character vs v2 BPE",
    "dataset": "Alice's Adventures in Wonderland",
    "validation_split": "Last 10% of the cleaned corpus",
    "character_bpc": char_bpc,
    "bpe_bpc": bpe_bpc,
    "relative_bpc_reduction_percent": (
        (char_bpc - bpe_bpc) / char_bpc * 100
    ),
    "caveat": (
        "Small model, one book, and limited training. "
        "Results do not establish general language capability."
    ),
}

summary_path = Path("/content/alice_model_comparison_summary.json")
summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

archive_path = Path("/content/alice_model_v1_v2_comparison.zip")

with zipfile.ZipFile(
    archive_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for label, folder in [
        ("character_baseline", v1_dir),
        ("bpe_v2", v2_dir),
    ]:
        for path in folder.rglob("*"):
            if path.is_file():
                archive.write(
                    path,
                    arcname=str(Path(label) / path.relative_to(folder)),
                )

    archive.write(comparison_path, arcname="alice_model_comparison.json")
    archive.write(
        summary_path,
        arcname="alice_model_comparison_summary.json",
    )

print("Character BPC:", round(char_bpc, 4))
print("BPE BPC:", round(bpe_bpc, 4))
print("Relative BPC reduction:", round(summary["relative_bpc_reduction_percent"], 2), "%")
print("Archive size:", f"{archive_path.stat().st_size:,}", "bytes")
print("SUCCESS: Both experiments and comparison reports are bundled.")

files.download(str(archive_path))


Character BPC: 2.682
BPE BPC: 2.2162
Relative BPC reduction: 17.37 %
Archive size: 12,633,320 bytes
SUCCESS: Both experiments and comparison reports are bundled.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
import os
import sys
import subprocess
from pathlib import Path

repo = Path("/content/ai-engineering-platform").resolve()

# تحديث المشروع إلى آخر التزام في GitHub
subprocess.run(["git", "fetch", "origin", "main"], cwd=repo, check=True)
subprocess.run(
    ["git", "checkout", "--force", "-B", "main", "origin/main"],
    cwd=repo,
    check=True,
)

os.chdir(repo)

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], text=True
).strip()
print("Commit:", commit)

assert commit == "25e9cb4", f"Expected 25e9cb4, got {commit}"

# تشغيل الاختبارات في عملية مستقلة لتجنب أي وحدات قديمة في ذاكرة Python
env = os.environ.copy()
env["PYTHONPATH"] = str(repo / "src") + os.pathsep + env.get("PYTHONPATH", "")

result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q", "-ra",
        "tests/test_language_model.py",
        "tests/test_language_model_tokenizer.py",
        "tests/test_language_model_bpe_tokenizer.py",
        "tests/test_language_model_training.py",
        "tests/test_language_model_evaluation.py",
    ],
    cwd=repo,
    env=env,
    text=True,
    capture_output=True,
)

print("\n--- TEST RESULTS ---")
print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Language-model tests failed; evaluation has not been accepted.")

print("SUCCESS: All language-model tests passed.")

Commit: 25e9cb4

--- TEST RESULTS ---
.................................                                        [100%]
33 passed in 3.07s

SUCCESS: All language-model tests passed.


In [22]:
import os
import sys
import json
import importlib
from pathlib import Path

import torch

repo = Path("/content/ai-engineering-platform").resolve()
os.chdir(repo)
sys.path.insert(0, str(repo / "src"))

assert torch.cuda.is_available(), "Tesla T4 غير متاحة حاليًا."

# تحميل أحدث أداة تقييم من المشروع
import ai_platform.machine_learning.language_model.training as training_module
import ai_platform.machine_learning.language_model.evaluation as evaluation_module

importlib.reload(training_module)
evaluation_module = importlib.reload(evaluation_module)

corpus_path = Path(
    "/content/model_lab_data/alice_adventures_in_wonderland.txt"
)
text = corpus_path.read_text(encoding="utf-8")
text = text.replace("\r\n", "\n").replace("\r", "\n")

# جزء التحقق نفسه المستخدم في التدريب
split_at = int(len(text) * 0.9)
validation_text = text[split_at:]

checkpoints = {
    "Character baseline": Path(
        "/content/alice_model_v1_character_clean/best.pt"
    ),
    "BPE v2": Path(
        "/content/alice_model_experiment_v2_bpe/best.pt"
    ),
}

results = {}

for name, checkpoint in checkpoints.items():
    assert checkpoint.is_file(), f"ملف النموذج غير موجود: {checkpoint}"

    metrics = evaluation_module.evaluate_language_model_checkpoint(
        checkpoint_path=checkpoint,
        validation_text=validation_text,
        device="cuda",
    )
    results[name] = metrics

    print(f"\n--- {name} ---")
    print(json.dumps(metrics, ensure_ascii=False, indent=2))

character_bpc = results["Character baseline"]["bits_per_character"]
bpe_bpc = results["BPE v2"]["bits_per_character"]

summary = {
    "character_bpc": character_bpc,
    "bpe_bpc": bpe_bpc,
    "relative_bpc_reduction_percent": (
        (character_bpc - bpe_bpc) / character_bpc * 100
    ),
    "validation_characters": len(validation_text),
    "method": "project evaluation.py on the same held-out text",
}

report = {
    "summary": summary,
    "results": results,
}

report_path = Path("/content/alice_model_comparison_project_evaluator.json")
report_path.write_text(
    json.dumps(report, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print("\n=== FINAL COMPARISON ===")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nReport saved to:", report_path)
print("SUCCESS: Both checkpoints evaluated with the project's evaluation utility.")



--- Character baseline ---
{
  "checkpoint": "/content/alice_model_v1_character_clean/best.pt",
  "checkpoint_step": 1000,
  "tokenizer_type": "character",
  "vocabulary_size": 79,
  "validation_characters": 14453,
  "validation_tokens": 14453,
  "predicted_tokens": 14452,
  "tokens_per_character": 1.0,
  "average_nll_per_token_nats": 1.8591357805773125,
  "bits_per_character": 2.681980392484315,
  "evaluation_method": "non-overlapping next-token blocks",
  "block_size": 128
}

--- BPE v2 ---
{
  "checkpoint": "/content/alice_model_experiment_v2_bpe/best.pt",
  "checkpoint_step": 1000,
  "tokenizer_type": "bpe_char",
  "vocabulary_size": 512,
  "validation_characters": 14453,
  "validation_tokens": 7758,
  "predicted_tokens": 7757,
  "tokens_per_character": 0.5367743721026776,
  "average_nll_per_token_nats": 2.8622045719824887,
  "bits_per_character": 2.216210452429439,
  "evaluation_method": "non-overlapping next-token blocks",
  "block_size": 128
}

=== FINAL COMPARISON ===
{
  "cha

In [23]:
import json
import zipfile
from pathlib import Path
from google.colab import files

v1_dir = Path("/content/alice_model_v1_character_clean")
v2_dir = Path("/content/alice_model_experiment_v2_bpe")
report_path = Path(
    "/content/alice_model_comparison_project_evaluator.json"
)

assert v1_dir.is_dir(), "لم يُعثر على ملفات Character baseline."
assert v2_dir.is_dir(), "لم يُعثر على ملفات BPE v2."
assert report_path.is_file(), "تقرير التقييم غير موجود."

report = json.loads(report_path.read_text(encoding="utf-8"))
results = report["results"]

summary = {
    "character_bpc": results["Character baseline"]["bits_per_character"],
    "bpe_bpc": results["BPE v2"]["bits_per_character"],
    "relative_bpc_reduction_percent": (
        (
            results["Character baseline"]["bits_per_character"]
            - results["BPE v2"]["bits_per_character"]
        )
        / results["Character baseline"]["bits_per_character"]
        * 100
    ),
    "validation_characters": results["Character baseline"]["validation_characters"],
    "validation_method": "Project evaluation utility",
    "caveat": "Results apply to this model, training setup, and validation split.",
}

summary_path = Path("/content/alice_model_final_comparison.json")
summary_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

archive_path = Path("/content/alice_model_v1_v2_final.zip")

with zipfile.ZipFile(
    archive_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for label, folder in [
        ("character_baseline", v1_dir),
        ("bpe_v2", v2_dir),
    ]:
        for path in folder.rglob("*"):
            if path.is_file():
                archive.write(
                    path,
                    arcname=str(Path(label) / path.relative_to(folder)),
                )

    archive.write(report_path, arcname="reports/project_evaluator_bpc.json")
    archive.write(summary_path, arcname="reports/final_comparison.json")

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Archive size:", f"{archive_path.stat().st_size:,}", "bytes")
print("SUCCESS: Both checkpoints and evaluation reports are bundled.")

files.download(str(archive_path))

{
  "character_bpc": 2.681980392484315,
  "bpe_bpc": 2.216210452429439,
  "relative_bpc_reduction_percent": 17.366642252870236,
  "validation_characters": 14453,
  "validation_method": "Project evaluation utility",
  "caveat": "Results apply to this model, training setup, and validation split."
}
Archive size: 12,632,749 bytes
SUCCESS: Both checkpoints and evaluation reports are bundled.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
from google.colab import files
uploaded = files.upload()

Saving pg1661.txt to pg1661.txt


In [25]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re
import unicodedata

# العثور على الملف الذي رفعته
matches = list(Path("/content").rglob("pg1661.txt"))
if not matches:
    raise FileNotFoundError(
        "لم يتم العثور على pg1661.txt داخل /content. أعد رفعه باستخدام files.upload()."
    )

source_path = matches[0]
raw_text = source_path.read_text(encoding="utf-8", errors="replace")
print("Source file:", source_path)
print("Original characters:", len(raw_text))

# تحديد بداية ونهاية متن الكتاب
start_match = re.search(
    r"\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK[^\n]*\*\*\*",
    raw_text,
    re.IGNORECASE,
)
end_match = re.search(
    r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK[^\n]*\*\*\*",
    raw_text,
    re.IGNORECASE,
)

if not start_match or not end_match or start_match.end() >= end_match.start():
    print("\nFirst lines:\n", "\n".join(raw_text.splitlines()[:20]))
    raise ValueError("لم أتمكن من تحديد علامتي البداية والنهاية بدقة.")

# استخراج المتن وتنظيف المسافات ونهايات الأسطر
clean_text = raw_text[start_match.end():end_match.start()]
clean_text = unicodedata.normalize("NFC", clean_text)
clean_text = clean_text.replace("\r\n", "\n").replace("\r", "\n")
clean_text = "\n".join(line.rstrip() for line in clean_text.splitlines())
clean_text = re.sub(r"\n{3,}", "\n\n", clean_text).strip() + "\n"

assert len(clean_text) > 100_000
assert "START OF THE PROJECT GUTENBERG EBOOK" not in clean_text
assert "END OF THE PROJECT GUTENBERG EBOOK" not in clean_text

output_dir = Path("/content/model_lab_data")
output_dir.mkdir(parents=True, exist_ok=True)

corpus_path = output_dir / "sherlock_holmes_external_test.txt"
metadata_path = output_dir / "sherlock_holmes_metadata.json"

corpus_path.write_text(clean_text, encoding="utf-8")

metadata = {
    "title": "The Adventures of Sherlock Holmes",
    "author": "Arthur Conan Doyle",
    "source_organization": "Project Gutenberg",
    "source_page": "https://www.gutenberg.org/ebooks/1661",
    "original_filename": "pg1661.txt",
    "purpose": "External evaluation only; not used for training or tokenizer fitting.",
    "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
    "character_count": len(clean_text),
    "word_count_approx": len(clean_text.split()),
    "line_count": len(clean_text.splitlines()),
    "sha256": hashlib.sha256(clean_text.encode("utf-8")).hexdigest(),
    "rights_note": (
        "Verify applicable local copyright law and the source's terms before "
        "redistribution or commercial use."
    ),
}

metadata_path.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print("\nSUCCESS: External test text cleaned.")
print("Clean text:", corpus_path)
print("Metadata:", metadata_path)
print("Characters:", metadata["character_count"])
print("Approximate words:", metadata["word_count_approx"])
print("SHA-256:", metadata["sha256"])

print("\n--- Text preview ---")
print(clean_text[:700])
print("\n--- End preview ---")
print(clean_text[-350:])

Source file: /content/ai-engineering-platform/pg1661.txt
Original characters: 581601

SUCCESS: External test text cleaned.
Clean text: /content/model_lab_data/sherlock_holmes_external_test.txt
Metadata: /content/model_lab_data/sherlock_holmes_metadata.json
Characters: 562138
Approximate words: 104506
SHA-256: e79b7c0c6341d620d5716eb48bb2631138c881409231aa5e26c20ed2fb79631a

--- Text preview ---
The Adventures of Sherlock Holmes

by Arthur Conan Doyle

Contents

   I.     A Scandal in Bohemia
   II.    The Red-Headed League
   III.   A Case of Identity
   IV.    The Boscombe Valley Mystery
   V.     The Five Orange Pips
   VI.    The Man with the Twisted Lip
   VII.   The Adventure of the Blue Carbuncle
   VIII.  The Adventure of the Speckled Band
   IX.    The Adventure of the Engineer’s Thumb
   X.     The Adventure of the Noble Bachelor
   XI.    The Adventure of the Beryl Coronet
   XII.   The Adventure of the Copper Beeches

I. A SCANDAL IN BOHEMIA

I.

To Sherlock Holmes she is al

In [26]:
import os
import sys
import json
import importlib
from pathlib import Path

import torch

repo = Path("/content/ai-engineering-platform").resolve()
os.chdir(repo)
sys.path.insert(0, str(repo / "src"))

assert torch.cuda.is_available(), "Tesla T4 GPU is not available."

# تحميل أداة التقييم الموجودة في المشروع
import ai_platform.machine_learning.language_model.training as training_module
import ai_platform.machine_learning.language_model.evaluation as evaluation_module

importlib.reload(training_module)
evaluation_module = importlib.reload(evaluation_module)

source_path = Path(
    "/content/model_lab_data/sherlock_holmes_external_test.txt"
)
assert source_path.is_file(), "ملف Sherlock المنظف غير موجود."

external_text = source_path.read_text(encoding="utf-8")

checkpoints = {
    "Character baseline": Path(
        "/content/alice_model_v1_character_clean/best.pt"
    ),
    "BPE v2": Path(
        "/content/alice_model_experiment_v2_bpe/best.pt"
    ),
}

results = {}

for name, checkpoint_path in checkpoints.items():
    assert checkpoint_path.is_file(), f"Checkpoint missing: {checkpoint_path}"

    # حساب BPC على الكتاب الخارجي كاملًا
    metrics = evaluation_module.evaluate_language_model_checkpoint(
        checkpoint_path=checkpoint_path,
        validation_text=external_text,
        device="cuda",
    )

    # قياس تغطية الرموز، دون إعادة تدريب الـTokenizer
    model, tokenizer, _ = training_module.load_language_model(
        checkpoint_path, device="cuda"
    )

    text_for_tokenizer = external_text
    if hasattr(tokenizer, "_normalize"):
        text_for_tokenizer = tokenizer._normalize(external_text)

    encoded = tokenizer.encode(text_for_tokenizer)
    unknown_id = tokenizer.unk_token_id
    unknown_count = sum(token_id == unknown_id for token_id in encoded)

    unknown_chars = sum(
        char not in tokenizer.token_to_id
        for char in text_for_tokenizer
    )

    metrics["unknown_tokens"] = unknown_count
    metrics["unknown_token_rate_percent"] = (
        100 * unknown_count / len(encoded) if encoded else 0.0
    )
    metrics["unknown_characters"] = unknown_chars
    metrics["unknown_character_rate_percent"] = (
        100 * unknown_chars / len(text_for_tokenizer)
        if text_for_tokenizer else 0.0
    )

    results[name] = metrics

    print(f"\n--- {name} ---")
    print(json.dumps(metrics, ensure_ascii=False, indent=2))

report = {
    "dataset": "The Adventures of Sherlock Holmes",
    "purpose": "External evaluation only; no training on this text.",
    "results": results,
}

report_path = Path("/content/sherlock_external_evaluation.json")
report_path.write_text(
    json.dumps(report, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print("\n=== EXTERNAL EVALUATION SUMMARY ===")
for name, metrics in results.items():
    print(
        f"{name}: BPC={metrics['bits_per_character']:.4f}, "
        f"UNK token rate={metrics['unknown_token_rate_percent']:.3f}%, "
        f"unknown character rate={metrics['unknown_character_rate_percent']:.3f}%"
    )

print("\nSUCCESS: External evaluation completed.")
print("Report saved to:", report_path)


--- Character baseline ---
{
  "checkpoint": "/content/alice_model_v1_character_clean/best.pt",
  "checkpoint_step": 1000,
  "tokenizer_type": "character",
  "vocabulary_size": 79,
  "validation_characters": 562138,
  "validation_tokens": 562138,
  "predicted_tokens": 562137,
  "tokens_per_character": 1.0,
  "average_nll_per_token_nats": 2.1128874859625557,
  "bits_per_character": 3.0482468753495846,
  "evaluation_method": "non-overlapping next-token blocks",
  "block_size": 128,
  "unknown_tokens": 284,
  "unknown_token_rate_percent": 0.05052140221796071,
  "unknown_characters": 284,
  "unknown_character_rate_percent": 0.05052140221796071
}

--- BPE v2 ---
{
  "checkpoint": "/content/alice_model_experiment_v2_bpe/best.pt",
  "checkpoint_step": 1000,
  "tokenizer_type": "bpe_char",
  "vocabulary_size": 512,
  "validation_characters": 562138,
  "validation_tokens": 324899,
  "predicted_tokens": 324898,
  "tokens_per_character": 0.5779701781413105,
  "average_nll_per_token_nats": 3.3420

In [27]:
import json
import zipfile
from pathlib import Path
from google.colab import files

v1_dir = Path("/content/alice_model_v1_character_clean")
v2_dir = Path("/content/alice_model_experiment_v2_bpe")

external_report_path = Path("/content/sherlock_external_evaluation.json")
alice_report_path = Path(
    "/content/alice_model_comparison_project_evaluator.json"
)
sherlock_metadata_path = Path(
    "/content/model_lab_data/sherlock_holmes_metadata.json"
)

for path in [
    v1_dir / "best.pt",
    v2_dir / "best.pt",
    external_report_path,
    alice_report_path,
    sherlock_metadata_path,
]:
    assert path.is_file(), f"Required artifact is missing: {path}"

external = json.loads(
    external_report_path.read_text(encoding="utf-8")
)
alice = json.loads(
    alice_report_path.read_text(encoding="utf-8")
)
sherlock_metadata = json.loads(
    sherlock_metadata_path.read_text(encoding="utf-8")
)

external_results = external["results"]
alice_results = alice["results"]

character_external = external_results["Character baseline"]["bits_per_character"]
bpe_external = external_results["BPE v2"]["bits_per_character"]

summary = {
    "project": "AI Engineering Platform",
    "experiments": {
        "alice_internal_validation": {
            "character_bpc": alice_results["Character baseline"]["bits_per_character"],
            "bpe_bpc": alice_results["BPE v2"]["bits_per_character"],
            "relative_bpc_reduction_percent": (
                (
                    alice_results["Character baseline"]["bits_per_character"]
                    - alice_results["BPE v2"]["bits_per_character"]
                )
                / alice_results["Character baseline"]["bits_per_character"]
                * 100
            ),
        },
        "sherlock_external_evaluation": {
            "character_bpc": character_external,
            "bpe_bpc": bpe_external,
            "relative_bpc_reduction_percent": (
                (character_external - bpe_external)
                / character_external
                * 100
            ),
            "validation_characters": (
                external_results["BPE v2"]["validation_characters"]
            ),
            "unknown_character_rate_percent": (
                external_results["BPE v2"]["unknown_character_rate_percent"]
            ),
        },
    },
    "limitations": [
        "Small transformer trained on one book for 1000 steps.",
        "External evaluation covers one additional English literary work.",
        "BPC comparisons apply to these checkpoints and datasets, not all tasks.",
    ],
}

summary_path = Path("/content/model_evaluation_summary.json")
summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

archive_path = Path("/content/ai_platform_language_model_evaluation.zip")

with zipfile.ZipFile(
    archive_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for label, folder in [
        ("character_baseline", v1_dir),
        ("bpe_v2", v2_dir),
    ]:
        for path in folder.rglob("*"):
            if path.is_file():
                archive.write(
                    path,
                    arcname=str(Path(label) / path.relative_to(folder)),
                )

    archive.write(
        external_report_path,
        arcname="reports/sherlock_external_evaluation.json",
    )
    archive.write(
        alice_report_path,
        arcname="reports/alice_internal_comparison.json",
    )
    archive.write(
        sherlock_metadata_path,
        arcname="reports/sherlock_dataset_metadata.json",
    )
    archive.write(
        summary_path,
        arcname="reports/model_evaluation_summary.json",
    )

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nArchive size:", f"{archive_path.stat().st_size:,}", "bytes")
print("SUCCESS: Models and evaluation reports bundled.")

files.download(str(archive_path))

{
  "project": "AI Engineering Platform",
  "experiments": {
    "alice_internal_validation": {
      "character_bpc": 2.681980392484315,
      "bpe_bpc": 2.216210452429439,
      "relative_bpc_reduction_percent": 17.366642252870236
    },
    "sherlock_external_evaluation": {
      "character_bpc": 3.0482468753495846,
      "bpe_bpc": 2.7867007464954323,
      "relative_bpc_reduction_percent": 8.580214777523791,
      "validation_characters": 562138,
      "unknown_character_rate_percent": 0.05052140221796071
    }
  },
  "limitations": [
    "Small transformer trained on one book for 1000 steps.",
    "External evaluation covers one additional English literary work.",
    "BPC comparisons apply to these checkpoints and datasets, not all tasks."
  ]
}

Archive size: 12,634,254 bytes
SUCCESS: Models and evaluation reports bundled.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
import os
import sys
import subprocess
from pathlib import Path

repo = Path("/content/ai-engineering-platform").resolve()

# تحديث المشروع إلى آخر نسخة على GitHub
subprocess.run(
    ["git", "fetch", "origin", "main"],
    cwd=repo,
    check=True,
)
subprocess.run(
    ["git", "checkout", "--force", "-B", "main", "origin/main"],
    cwd=repo,
    check=True,
)

os.chdir(repo)

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=repo,
    text=True,
).strip()

print("Commit:", commit)
assert commit == "22f172c", f"Expected 22f172c, received {commit}"

# تشغيل اختبارات مكوّن إدارة البيانات
env = os.environ.copy()
env["PYTHONPATH"] = (
    str(repo / "src") + os.pathsep + env.get("PYTHONPATH", "")
)

result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_language_model_corpus.py",
    ],
    cwd=repo,
    env=env,
    text=True,
    capture_output=True,
)

print("\n--- CORPUS TEST RESULTS ---")
print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Corpus tests failed.")

print("SUCCESS: Corpus manager is ready.")

FileNotFoundError: [Errno 2] No such file or directory: PosixPath('/content/ai-engineering-platform')

In [2]:
from pathlib import Path
import subprocess
import os

repo = Path("/content/ai-engineering-platform")

if not repo.exists():
    print("Project folder is missing. Restoring from GitHub...")
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/F13342/ai-engineering-platform.git",
            str(repo),
        ],
        check=True,
    )
else:
    print("Project folder exists. Updating from GitHub...")
    subprocess.run(
        ["git", "fetch", "origin", "main"],
        cwd=repo,
        check=True,
    )
    subprocess.run(
        ["git", "checkout", "--force", "-B", "main", "origin/main"],
        cwd=repo,
        check=True,
    )

os.chdir(repo)

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=repo,
    text=True,
).strip()

print("SUCCESS: Project restored.")
print("Current commit:", commit)
print("Project directory:", repo)

Project folder is missing. Restoring from GitHub...
SUCCESS: Project restored.
Current commit: 54b97b1
Project directory: /content/ai-engineering-platform


In [3]:
import subprocess
import os
from pathlib import Path

repo = Path("/content/ai-engineering-platform")

# جلب أحدث نسخة من الفرع main
subprocess.run(
    ["git", "fetch", "origin", "main"],
    cwd=repo,
    check=True,
)

# الانتقال صراحةً إلى main، بدل الفرع الافتراضي الذي استُنسخ
subprocess.run(
    ["git", "checkout", "--force", "-B", "main", "origin/main"],
    cwd=repo,
    check=True,
)

os.chdir(repo)

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=repo,
    text=True,
).strip()

print("Current commit:", commit)
print("Corpus manager exists:",
      (repo / "src/ai_platform/machine_learning/language_model/corpus.py").is_file())
print("Evaluation module exists:",
      (repo / "src/ai_platform/machine_learning/language_model/evaluation.py").is_file())
print("BPE tokenizer exists:",
      (repo / "src/ai_platform/machine_learning/language_model/bpe_tokenizer.py").is_file())

assert commit == "22f172c", f"Expected latest commit 22f172c, got {commit}"
assert (repo / "src/ai_platform/machine_learning/language_model/corpus.py").is_file()
assert (repo / "src/ai_platform/machine_learning/language_model/evaluation.py").is_file()
assert (repo / "src/ai_platform/machine_learning/language_model/bpe_tokenizer.py").is_file()

print("SUCCESS: Latest project version restored.")

Current commit: 22f172c
Corpus manager exists: True
Evaluation module exists: True
BPE tokenizer exists: True
SUCCESS: Latest project version restored.


In [4]:
import os
import sys
import subprocess
from pathlib import Path

repo = Path("/content/ai-engineering-platform").resolve()
os.chdir(repo)

env = os.environ.copy()
env["PYTHONPATH"] = (
    str(repo / "src") + os.pathsep + env.get("PYTHONPATH", "")
)

print("Project commit:", subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=repo,
    text=True,
).strip())

result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_language_model_corpus.py",
    ],
    cwd=repo,
    env=env,
    capture_output=True,
    text=True,
)

print("\n--- CORPUS TEST RESULTS ---")
print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Corpus manager tests failed.")

print("SUCCESS: Corpus manager is ready.")

Project commit: 22f172c

--- CORPUS TEST RESULTS ---
........                                                                 [100%]
8 passed in 0.04s

SUCCESS: Corpus manager is ready.


In [5]:
from pathlib import Path
import subprocess

paths = [
    "/content/pg11.txt",
    "/content/pg1661.txt",
    "/content/model_lab_data/alice_adventures_in_wonderland.txt",
    "/content/model_lab_data/sherlock_holmes_external_test.txt",
    "/content/alice_model_v1_character_clean/best.pt",
    "/content/alice_model_experiment_v2_bpe/best.pt",
    "/content/ai_platform_language_model_evaluation.zip",
]

print("=== Available files ===")
for item in paths:
    path = Path(item)
    if path.is_file():
        print(f"[FOUND] {item} ({path.stat().st_size:,} bytes)")
    else:
        print(f"[MISSING] {item}")

repo = Path("/content/ai-engineering-platform")
if repo.is_dir():
    commit = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=repo,
        text=True,
    ).strip()
    print("\nProject commit:", commit)

try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("PyTorch is not installed in this runtime.")

=== Available files ===
[MISSING] /content/pg11.txt
[MISSING] /content/pg1661.txt
[MISSING] /content/model_lab_data/alice_adventures_in_wonderland.txt
[MISSING] /content/model_lab_data/sherlock_holmes_external_test.txt
[MISSING] /content/alice_model_v1_character_clean/best.pt
[MISSING] /content/alice_model_experiment_v2_bpe/best.pt
[MISSING] /content/ai_platform_language_model_evaluation.zip

Project commit: 22f172c
CUDA available: True
GPU: Tesla T4


In [6]:
import os
import sys
import json
import hashlib
import urllib.request
from pathlib import Path

repo = Path("/content/ai-engineering-platform").resolve()
os.chdir(repo)
sys.path.insert(0, str(repo / "src"))

from ai_platform.machine_learning.language_model.corpus import build_corpus

# تنزيل ملفات النصوص الأصلية من Project Gutenberg
sources_dir = Path("/content/model_lab_source_files")
output_dir = Path("/content/model_lab_data")
sources_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

books = [
    {
        "source_id": "alice_11",
        "title": "Alice's Adventures in Wonderland",
        "author": "Lewis Carroll",
        "ebook_id": "11",
        "split": "train",
    },
    {
        "source_id": "frankenstein_84",
        "title": "Frankenstein",
        "author": "Mary Wollstonecraft Shelley",
        "ebook_id": "84",
        "split": "train",
    },
    {
        "source_id": "pride_prejudice_1342",
        "title": "Pride and Prejudice",
        "author": "Jane Austen",
        "ebook_id": "1342",
        "split": "train",
    },
    {
        "source_id": "sherlock_holmes_1661",
        "title": "The Adventures of Sherlock Holmes",
        "author": "Arthur Conan Doyle",
        "ebook_id": "1661",
        "split": "test",
    },
]

records = []

for book in books:
    ebook_id = book["ebook_id"]
    url = f"https://www.gutenberg.org/cache/epub/{ebook_id}/pg{ebook_id}.txt"
    target = sources_dir / f"pg{ebook_id}.txt"

    if not target.is_file():
        request = urllib.request.Request(
            url,
            headers={"User-Agent": "AI-Engineering-Platform/1.0 research corpus preparation"},
        )
        with urllib.request.urlopen(request, timeout=60) as response:
            raw = response.read()

        # تحقق من أن الاستجابة نص UTF-8 صالح
        raw.decode("utf-8-sig")
        target.write_bytes(raw)

    raw_bytes = target.read_bytes()
    raw_text = raw_bytes.decode("utf-8-sig")

    print(
        f"Ready: {book['title']} | "
        f"{len(raw_text):,} characters | SHA-256 "
        f"{hashlib.sha256(raw_bytes).hexdigest()[:16]}..."
    )

    records.append({
        "source_id": book["source_id"],
        "path": str(target),
        "title": book["title"],
        "author": book["author"],
        "source_url": f"https://www.gutenberg.org/ebooks/{ebook_id}",
        "license_name": (
            "Project Gutenberg source; public-domain status in the US, "
            "local legal review required"
        ),
        "license_url": "https://www.gutenberg.org/policy/terms_of_use",
        "rights_status": "research_only",
        "split": book["split"],
    })

# إنشاء مجموعة التدريب فقط وسجل مصادرها
train_manifest = build_corpus(
    records,
    output_dir / "multi_source_train.txt",
    output_dir / "multi_source_train_manifest.json",
    target_split="train",
    purpose="research",
)

# إنشاء مجموعة الاختبار الخارجية مستقلة عن التدريب
test_manifest = build_corpus(
    records,
    output_dir / "sherlock_external_test.txt",
    output_dir / "sherlock_external_test_manifest.json",
    target_split="test",
    purpose="research",
)

print("\n=== DATASET SUMMARY ===")
print("Training sources:", train_manifest["source_count"])
print("Training characters:", f'{train_manifest["character_count"]:,}')
print("Training SHA-256:", train_manifest["corpus_sha256"])
print("External test sources:", test_manifest["source_count"])
print("External test characters:", f'{test_manifest["character_count"]:,}')
print("External test SHA-256:", test_manifest["corpus_sha256"])

print("\nTraining books:")
for source in train_manifest["sources"]:
    print("-", source["title"], "|", source["rights_status"])

print("\nExternal test books:")
for source in test_manifest["sources"]:
    print("-", source["title"], "|", source["rights_status"])

assert train_manifest["source_count"] == 3
assert test_manifest["source_count"] == 1
assert "Sherlock Holmes" in test_manifest["sources"][0]["title"]

print("\nSUCCESS: Multi-source training corpus and isolated test set created.")

Ready: Alice's Adventures in Wonderland | 167,712 characters | SHA-256 01b38ea4c710a84b...
Ready: Frankenstein | 446,582 characters | SHA-256 7810cd483cffcf2c...
Ready: Pride and Prejudice | 763,082 characters | SHA-256 3f6bb9d6f78e0293...
Ready: The Adventures of Sherlock Holmes | 593,911 characters | SHA-256 922e2a12ccb43a4c...

=== DATASET SUMMARY ===
Training sources: 3
Training characters: 1,291,986
Training SHA-256: c5e4b94a58a77a5e5ad2db4ad96b113eb26bc134fb572a2ac9a5a0d093db7ddb
External test sources: 1
External test characters: 562,138
External test SHA-256: e79b7c0c6341d620d5716eb48bb2631138c881409231aa5e26c20ed2fb79631a

Training books:
- Alice's Adventures in Wonderland | research_only
- Frankenstein | research_only
- Pride and Prejudice | research_only

External test books:
- The Adventures of Sherlock Holmes | research_only

SUCCESS: Multi-source training corpus and isolated test set created.


In [7]:
import os
import sys
import json
import shutil
import importlib
from pathlib import Path

import torch

repo = Path("/content/ai-engineering-platform").resolve()
os.chdir(repo)
sys.path.insert(0, str(repo / "src"))

import ai_platform.machine_learning.language_model.training as training_module

importlib.invalidate_caches()
training_module = importlib.reload(training_module)

# الملفات التي جهزناها في الخطوة السابقة
data_dir = Path("/content/model_lab_data")
train_path = data_dir / "multi_source_train.txt"
train_manifest_path = data_dir / "multi_source_train_manifest.json"
test_path = data_dir / "sherlock_external_test.txt"
test_manifest_path = data_dir / "sherlock_external_test_manifest.json"

for path in [
    train_path,
    train_manifest_path,
    test_path,
    test_manifest_path,
]:
    assert path.is_file(), f"Required file is missing: {path}"

assert torch.cuda.is_available(), "Tesla T4 is not available."

train_manifest = json.loads(
    train_manifest_path.read_text(encoding="utf-8")
)
test_manifest = json.loads(
    test_manifest_path.read_text(encoding="utf-8")
)

train_ids = {item["source_id"] for item in train_manifest["sources"]}
test_ids = {item["source_id"] for item in test_manifest["sources"]}

assert train_ids.isdisjoint(test_ids), (
    "Training and external-test sources must remain separate."
)

print("GPU:", torch.cuda.get_device_name(0))
print("Training sources:", train_ids)
print("External test sources:", test_ids)
print("Training characters:", f"{len(train_path.read_text(encoding='utf-8')):,}")
print("External test characters:", f"{len(test_path.read_text(encoding='utf-8')):,}")

# تجربة v3 — نفس حجم النموذج وعدد الخطوات المستخدم في v2
output_dir = Path("/content/alice_model_v3_multisource_bpe")

config = training_module.LanguageModelTrainingConfig(
    block_size=128,
    n_layers=4,
    n_heads=4,
    n_embd=128,
    dropout=0.1,
    batch_size=32,
    max_steps=1000,
    learning_rate=3e-4,
    weight_decay=0.01,
    eval_interval=100,
    eval_batches=10,
    train_fraction=0.9,
    grad_clip=1.0,
    seed=42,
    tokenizer_type="bpe",
    tokenizer_vocab_size=512,
    tokenizer_min_frequency=2,
    device="cuda",
)

result = training_module.train_language_model(
    corpus_path=train_path,
    output_dir=output_dir,
    config=config,
)

# حفظ بيانات التجربة ومصادرها
shutil.copy2(
    train_manifest_path,
    output_dir / "training_dataset_manifest.json",
)
shutil.copy2(
    test_manifest_path,
    output_dir / "external_test_manifest.json",
)

experiment = {
    "experiment": "Alice Model v3 — Multi-source BPE",
    "initialization": "random",
    "training_sources": sorted(train_ids),
    "held_out_external_test_sources": sorted(test_ids),
    "training_config": config.__dict__,
    "training_results": result,
    "note": (
        "The trainer uses an internal contiguous 90/10 text split for "
        "training and validation. Sherlock Holmes remains excluded from training."
    ),
}

(output_dir / "experiment.json").write_text(
    json.dumps(experiment, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print("\n=== ALICE MODEL V3 SUMMARY ===")
print(json.dumps(result, ensure_ascii=False, indent=2))

for name in [
    "best.pt",
    "last.pt",
    "tokenizer.json",
    "metrics.json",
    "training_dataset_manifest.json",
    "external_test_manifest.json",
    "experiment.json",
]:
    artifact = output_dir / name
    assert artifact.is_file(), f"Missing artifact: {name}"
    print(f"Verified {name}: {artifact.stat().st_size:,} bytes")

print("\nSUCCESS: Multi-source v3 training completed.")
print("Artifacts:", output_dir)

GPU: Tesla T4
Training sources: {'frankenstein_84', 'alice_11', 'pride_prejudice_1342'}
External test sources: {'sherlock_holmes_1661'}
Training characters: 1,291,986
External test characters: 562,138
step 100/1000 | train_loss=4.0015 | val_loss=3.9812 | device=cuda
step 200/1000 | train_loss=3.6817 | val_loss=3.6680 | device=cuda
step 300/1000 | train_loss=3.5210 | val_loss=3.4711 | device=cuda
step 400/1000 | train_loss=3.3957 | val_loss=3.3512 | device=cuda
step 500/1000 | train_loss=3.2990 | val_loss=3.2435 | device=cuda
step 600/1000 | train_loss=3.2234 | val_loss=3.1760 | device=cuda
step 700/1000 | train_loss=3.1660 | val_loss=3.1107 | device=cuda
step 800/1000 | train_loss=3.1026 | val_loss=3.0592 | device=cuda
step 900/1000 | train_loss=3.0396 | val_loss=2.9712 | device=cuda
step 1000/1000 | train_loss=2.9693 | val_loss=2.9123 | device=cuda

=== ALICE MODEL V3 SUMMARY ===
{
  "output_dir": "/content/alice_model_v3_multisource_bpe",
  "best_checkpoint": "/content/alice_model_v3

In [8]:
import os
import sys
import json
import hashlib
from pathlib import Path

import torch

repo = Path("/content/ai-engineering-platform").resolve()
os.chdir(repo)
sys.path.insert(0, str(repo / "src"))

assert torch.cuda.is_available(), "Tesla T4 is not available."

from ai_platform.machine_learning.language_model.training import (
    load_language_model,
    generate_text,
)
from ai_platform.machine_learning.language_model.evaluation import (
    evaluate_language_model_checkpoint,
)

# البيانات الخارجية التي لم تدخل في التدريب
test_path = Path("/content/model_lab_data/sherlock_external_test.txt")
checkpoint_path = Path(
    "/content/alice_model_v3_multisource_bpe/best.pt"
)
metadata_path = Path(
    "/content/model_lab_data/sherlock_external_test_manifest.json"
)

for path in [test_path, checkpoint_path, metadata_path]:
    assert path.is_file(), f"Required file missing: {path}"

text = test_path.read_text(encoding="utf-8")
test_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()

print("GPU:", torch.cuda.get_device_name(0))
print("V3 checkpoint:", checkpoint_path)
print("External test characters:", len(text))
print("External test SHA-256:", test_hash)

# تقييم v3 باستخدام أداة المشروع الدائمة
metrics = evaluate_language_model_checkpoint(
    checkpoint_path=checkpoint_path,
    validation_text=text,
    device="cuda",
)

# قياس نسبة الرموز والمحارف غير الموجودة في مفردات v3
model, tokenizer, checkpoint = load_language_model(
    checkpoint_path,
    device="cuda",
)

text_for_tokenizer = tokenizer._normalize(text)
encoded = tokenizer.encode(text_for_tokenizer)

unknown_id = tokenizer.unk_token_id
unknown_tokens = sum(token_id == unknown_id for token_id in encoded)
unknown_characters = sum(
    char not in tokenizer.token_to_id for char in text_for_tokenizer
)

metrics["unknown_tokens"] = unknown_tokens
metrics["unknown_token_rate_percent"] = (
    100 * unknown_tokens / len(encoded) if encoded else 0.0
)
metrics["unknown_characters"] = unknown_characters
metrics["unknown_character_rate_percent"] = (
    100 * unknown_characters / len(text_for_tokenizer)
    if text_for_tokenizer else 0.0
)

# أرقام v1 وv2 من التقييم السابق على النص نفسه
historical_bpc = {
    "Alice v1 Character": 3.0482468753495846,
    "Alice v2 BPE": 2.7867007464954323,
}

comparison = {
    "historical_bpc": historical_bpc,
    "v3_bpc": metrics["bits_per_character"],
    "relative_reduction_vs_v2_percent": (
        (
            historical_bpc["Alice v2 BPE"]
            - metrics["bits_per_character"]
        )
        / historical_bpc["Alice v2 BPE"]
        * 100
    ),
    "v3_metrics": metrics,
    "external_test_sha256": test_hash,
}

print("\n=== EXTERNAL TEST RESULTS ===")
print(json.dumps(comparison, ensure_ascii=False, indent=2))

# عينة نوعية؛ لا تُستخدم هذه العينة لحساب BPC
sample = generate_text(
    checkpoint_path,
    prompt="To Sherlock Holmes",
    max_new_tokens=180,
    temperature=0.8,
    top_k=20,
    device="cuda",
)

print("\n=== V3 GENERATED SAMPLE ===")
print(sample)

report_path = Path("/content/sherlock_v3_evaluation.json")
report_path.write_text(
    json.dumps(comparison, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print("\nSUCCESS: V3 external evaluation completed.")
print("Report:", report_path)

GPU: Tesla T4
V3 checkpoint: /content/alice_model_v3_multisource_bpe/best.pt
External test characters: 562138
External test SHA-256: e79b7c0c6341d620d5716eb48bb2631138c881409231aa5e26c20ed2fb79631a

=== EXTERNAL TEST RESULTS ===
{
  "historical_bpc": {
    "Alice v1 Character": 3.0482468753495846,
    "Alice v2 BPE": 2.7867007464954323
  },
  "v3_bpc": 2.5875072126468677,
  "relative_reduction_vs_v2_percent": 7.148005902645677,
  "v3_metrics": {
    "checkpoint": "/content/alice_model_v3_multisource_bpe/best.pt",
    "checkpoint_step": 1000,
    "tokenizer_type": "bpe_char",
    "vocabulary_size": 512,
    "validation_characters": 562138,
    "validation_tokens": 323197,
    "predicted_tokens": 323196,
    "tokens_per_character": 0.5749424518534595,
    "average_nll_per_token_nats": 3.119492868684947,
    "bits_per_character": 2.5875072126468677,
    "evaluation_method": "non-overlapping next-token blocks",
    "block_size": 128,
    "unknown_tokens": 38,
    "unknown_token_rate_percen

In [9]:
import os
import sys
import json
import shutil
import zipfile
from pathlib import Path
from google.colab import files

repo = Path("/content/ai-engineering-platform").resolve()
sys.path.insert(0, str(repo / "src"))

v3_dir = Path("/content/alice_model_v3_multisource_bpe")
data_dir = Path("/content/model_lab_data")
source_dir = Path("/content/model_lab_source_files")

external_report = Path("/content/sherlock_v3_evaluation.json")

required = [
    v3_dir / "best.pt",
    v3_dir / "last.pt",
    v3_dir / "tokenizer.json",
    v3_dir / "metrics.json",
    v3_dir / "training_dataset_manifest.json",
    v3_dir / "external_test_manifest.json",
    v3_dir / "experiment.json",
    data_dir / "multi_source_train.txt",
    data_dir / "sherlock_external_test.txt",
    external_report,
]

for path in required:
    assert path.is_file(), f"Required artifact is missing: {path}"

backup = Path("/content/ai_platform_v3_backup")
if backup.exists():
    shutil.rmtree(backup)

(backup / "model").mkdir(parents=True)
(backup / "data" / "source_files").mkdir(parents=True)
(backup / "data" / "cleaned").mkdir(parents=True)
(backup / "reports").mkdir(parents=True)

# 1. نسخة كاملة من مخرجات v3
shutil.copytree(v3_dir, backup / "model" / "artifacts")

# 2. مجموعات النصوص المستخدمة للاختبار والتدريب
shutil.copy2(
    data_dir / "multi_source_train.txt",
    backup / "data" / "cleaned" / "multi_source_train.txt",
)
shutil.copy2(
    data_dir / "sherlock_external_test.txt",
    backup / "data" / "cleaned" / "sherlock_external_test.txt",
)

# 3. حفظ سجل المصدرين مع مسارات نسبية قابلة للنقل
for filename, output_name in [
    ("multi_source_train_manifest.json", "training_manifest.json"),
    ("sherlock_external_test_manifest.json", "external_test_manifest.json"),
]:
    source_manifest_path = data_dir / filename
    manifest = json.loads(source_manifest_path.read_text(encoding="utf-8"))

    for item in manifest.get("sources", []):
        item["path"] = "data/source_files/" + Path(item["path"]).name

    (backup / "data" / output_name).write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

# 4. نسخ ملفات المصدر الأصلية، إن كانت متاحة في هذه الجلسة
source_names = ["pg11.txt", "pg84.txt", "pg1342.txt", "pg1661.txt"]
for name in source_names:
    original = source_dir / name
    assert original.is_file(), f"Original source file is missing: {original}"
    shutil.copy2(original, backup / "data" / "source_files" / name)

# 5. حفظ تقرير التقييم الخارجي وملخص المقارنة
report = json.loads(external_report.read_text(encoding="utf-8"))
(backup / "reports" / "sherlock_v3_evaluation.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

historical = report["historical_bpc"]
v3_bpc = report["v3_metrics"]["bits_per_character"]
comparison = {
    "model": "Alice Model v3 — Multi-source BPE",
    "external_dataset": "The Adventures of Sherlock Holmes",
    "validation_characters": report["v3_metrics"]["validation_characters"],
    "character_baseline_bpc": historical["Alice v1 Character"],
    "alice_v2_bpc": historical["Alice v2 BPE"],
    "alice_v3_bpc": v3_bpc,
    "relative_reduction_vs_v1_percent": (
        (historical["Alice v1 Character"] - v3_bpc)
        / historical["Alice v1 Character"] * 100
    ),
    "relative_reduction_vs_v2_percent": (
        (historical["Alice v2 BPE"] - v3_bpc)
        / historical["Alice v2 BPE"] * 100
    ),
    "unknown_characters": report["v3_metrics"]["unknown_characters"],
    "unknown_character_rate_percent": (
        report["v3_metrics"]["unknown_character_rate_percent"]
    ),
    "limitations": [
        "The model was trained for 1000 steps on three English literary books.",
        "Sherlock Holmes is an external test set, but its results have been used for model comparisons.",
        "Low BPC does not establish general conversational or reasoning ability.",
        "Research-only source status is not legal clearance for commercial use.",
    ],
}

(backup / "reports" / "comparison_summary.json").write_text(
    json.dumps(comparison, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

readme = """AI Engineering Platform — Alice Model v3 Backup

Contents
- model/artifacts/: v3 model checkpoints, BPE tokenizer, training metrics,
  training manifest, external-test manifest, and experiment configuration.
- data/cleaned/: exact training corpus and external-test text used in this run.
- data/source_files/: original Project Gutenberg source text files.
- data/: portable manifests with relative source paths.
- reports/: Sherlock external evaluation and comparison summary.

Reproduction notes
- Model architecture and training utilities are stored in the GitHub project.
- The v3 model was initialized randomly and trained on three English books.
- Sherlock Holmes was held out from training.
- The external evaluation has already been used for model comparisons; use a
  new, untouched test corpus for final model-selection claims.

Rights note
- Project Gutenberg's US public-domain classification does not guarantee
  availability in every jurisdiction. Verify local law and each source's
  terms before redistribution or commercial use.
- Source records in this experiment are marked research_only.
"""

(backup / "README.txt").write_text(readme, encoding="utf-8")

archive_path = Path("/content/ai_platform_v3_multisource_backup.zip")
if archive_path.exists():
    archive_path.unlink()

with zipfile.ZipFile(
    archive_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in sorted(backup.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(backup).as_posix())

print("=== V3 BACKUP SUMMARY ===")
print(json.dumps(comparison, ensure_ascii=False, indent=2))
print("\nArchive:", archive_path)
print("Archive size:", f"{archive_path.stat().st_size:,}", "bytes")
print("SUCCESS: v3 model, data, manifests, and reports are bundled.")

files.download(str(archive_path))

=== V3 BACKUP SUMMARY ===
{
  "model": "Alice Model v3 — Multi-source BPE",
  "external_dataset": "The Adventures of Sherlock Holmes",
  "validation_characters": 562138,
  "character_baseline_bpc": 3.0482468753495846,
  "alice_v2_bpc": 2.7867007464954323,
  "alice_v3_bpc": 2.5875072126468677,
  "relative_reduction_vs_v1_percent": 15.11490642141239,
  "relative_reduction_vs_v2_percent": 7.148005902645677,
  "unknown_characters": 38,
  "unknown_character_rate_percent": 0.006759905930572208,
  "limitations": [
    "The model was trained for 1000 steps on three English literary books.",
    "Sherlock Holmes is an external test set, but its results have been used for model comparisons.",
    "Low BPC does not establish general conversational or reasoning ability.",
    "Research-only source status is not legal clearance for commercial use."
  ]
}

Archive: /content/ai_platform_v3_multisource_backup.zip
Archive size: 7,960,130 bytes
SUCCESS: v3 model, data, manifests, and reports are bundled

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
import os
import sys
import json
import shutil
import zipfile
from pathlib import Path
from google.colab import files

repo = Path("/content/ai-engineering-platform").resolve()
sys.path.insert(0, str(repo / "src"))

v3_dir = Path("/content/alice_model_v3_multisource_bpe")
data_dir = Path("/content/model_lab_data")
source_dir = Path("/content/model_lab_source_files")

external_report = Path("/content/sherlock_v3_evaluation.json")

required = [
    v3_dir / "best.pt",
    v3_dir / "last.pt",
    v3_dir / "tokenizer.json",
    v3_dir / "metrics.json",
    v3_dir / "training_dataset_manifest.json",
    v3_dir / "external_test_manifest.json",
    v3_dir / "experiment.json",
    data_dir / "multi_source_train.txt",
    data_dir / "sherlock_external_test.txt",
    external_report,
]

for path in required:
    assert path.is_file(), f"Required artifact is missing: {path}"

backup = Path("/content/ai_platform_v3_backup")
if backup.exists():
    shutil.rmtree(backup)

(backup / "model").mkdir(parents=True)
(backup / "data" / "source_files").mkdir(parents=True)
(backup / "data" / "cleaned").mkdir(parents=True)
(backup / "reports").mkdir(parents=True)

# 1. نسخة كاملة من مخرجات v3
shutil.copytree(v3_dir, backup / "model" / "artifacts")

# 2. مجموعات النصوص المستخدمة للاختبار والتدريب
shutil.copy2(
    data_dir / "multi_source_train.txt",
    backup / "data" / "cleaned" / "multi_source_train.txt",
)
shutil.copy2(
    data_dir / "sherlock_external_test.txt",
    backup / "data" / "cleaned" / "sherlock_external_test.txt",
)

# 3. حفظ سجل المصدرين مع مسارات نسبية قابلة للنقل
for filename, output_name in [
    ("multi_source_train_manifest.json", "training_manifest.json"),
    ("sherlock_external_test_manifest.json", "external_test_manifest.json"),
]:
    source_manifest_path = data_dir / filename
    manifest = json.loads(source_manifest_path.read_text(encoding="utf-8"))

    for item in manifest.get("sources", []):
        item["path"] = "data/source_files/" + Path(item["path"]).name

    (backup / "data" / output_name).write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

# 4. نسخ ملفات المصدر الأصلية، إن كانت متاحة في هذه الجلسة
source_names = ["pg11.txt", "pg84.txt", "pg1342.txt", "pg1661.txt"]
for name in source_names:
    original = source_dir / name
    assert original.is_file(), f"Original source file is missing: {original}"
    shutil.copy2(original, backup / "data" / "source_files" / name)

# 5. حفظ تقرير التقييم الخارجي وملخص المقارنة
report = json.loads(external_report.read_text(encoding="utf-8"))
(backup / "reports" / "sherlock_v3_evaluation.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

historical = report["historical_bpc"]
v3_bpc = report["v3_metrics"]["bits_per_character"]
comparison = {
    "model": "Alice Model v3 — Multi-source BPE",
    "external_dataset": "The Adventures of Sherlock Holmes",
    "validation_characters": report["v3_metrics"]["validation_characters"],
    "character_baseline_bpc": historical["Alice v1 Character"],
    "alice_v2_bpc": historical["Alice v2 BPE"],
    "alice_v3_bpc": v3_bpc,
    "relative_reduction_vs_v1_percent": (
        (historical["Alice v1 Character"] - v3_bpc)
        / historical["Alice v1 Character"] * 100
    ),
    "relative_reduction_vs_v2_percent": (
        (historical["Alice v2 BPE"] - v3_bpc)
        / historical["Alice v2 BPE"] * 100
    ),
    "unknown_characters": report["v3_metrics"]["unknown_characters"],
    "unknown_character_rate_percent": (
        report["v3_metrics"]["unknown_character_rate_percent"]
    ),
    "limitations": [
        "The model was trained for 1000 steps on three English literary books.",
        "Sherlock Holmes is an external test set, but its results have been used for model comparisons.",
        "Low BPC does not establish general conversational or reasoning ability.",
        "Research-only source status is not legal clearance for commercial use.",
    ],
}

(backup / "reports" / "comparison_summary.json").write_text(
    json.dumps(comparison, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

readme = """AI Engineering Platform — Alice Model v3 Backup

Contents
- model/artifacts/: v3 model checkpoints, BPE tokenizer, training metrics,
  training manifest, external-test manifest, and experiment configuration.
- data/cleaned/: exact training corpus and external-test text used in this run.
- data/source_files/: original Project Gutenberg source text files.
- data/: portable manifests with relative source paths.
- reports/: Sherlock external evaluation and comparison summary.

Reproduction notes
- Model architecture and training utilities are stored in the GitHub project.
- The v3 model was initialized randomly and trained on three English books.
- Sherlock Holmes was held out from training.
- The external evaluation has already been used for model comparisons; use a
  new, untouched test corpus for final model-selection claims.

Rights note
- Project Gutenberg's US public-domain classification does not guarantee
  availability in every jurisdiction. Verify local law and each source's
  terms before redistribution or commercial use.
- Source records in this experiment are marked research_only.
"""

(backup / "README.txt").write_text(readme, encoding="utf-8")

archive_path = Path("/content/ai_platform_v3_multisource_backup.zip")
if archive_path.exists():
    archive_path.unlink()

with zipfile.ZipFile(
    archive_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in sorted(backup.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(backup).as_posix())

print("=== V3 BACKUP SUMMARY ===")
print(json.dumps(comparison, ensure_ascii=False, indent=2))
print("\nArchive:", archive_path)
print("Archive size:", f"{archive_path.stat().st_size:,}", "bytes")
print("SUCCESS: v3 model, data, manifests, and reports are bundled.")

files.download(str(archive_path))

=== V3 BACKUP SUMMARY ===
{
  "model": "Alice Model v3 — Multi-source BPE",
  "external_dataset": "The Adventures of Sherlock Holmes",
  "validation_characters": 562138,
  "character_baseline_bpc": 3.0482468753495846,
  "alice_v2_bpc": 2.7867007464954323,
  "alice_v3_bpc": 2.5875072126468677,
  "relative_reduction_vs_v1_percent": 15.11490642141239,
  "relative_reduction_vs_v2_percent": 7.148005902645677,
  "unknown_characters": 38,
  "unknown_character_rate_percent": 0.006759905930572208,
  "limitations": [
    "The model was trained for 1000 steps on three English literary books.",
    "Sherlock Holmes is an external test set, but its results have been used for model comparisons.",
    "Low BPC does not establish general conversational or reasoning ability.",
    "Research-only source status is not legal clearance for commercial use."
  ]
}

Archive: /content/ai_platform_v3_multisource_backup.zip
Archive size: 7,960,130 bytes
SUCCESS: v3 model, data, manifests, and reports are bundled

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
import os
import sys
import json
import shutil
import zipfile
from pathlib import Path
from google.colab import files

repo = Path("/content/ai-engineering-platform").resolve()
sys.path.insert(0, str(repo / "src"))

v3_dir = Path("/content/alice_model_v3_multisource_bpe")
data_dir = Path("/content/model_lab_data")
source_dir = Path("/content/model_lab_source_files")

external_report = Path("/content/sherlock_v3_evaluation.json")

required = [
    v3_dir / "best.pt",
    v3_dir / "last.pt",
    v3_dir / "tokenizer.json",
    v3_dir / "metrics.json",
    v3_dir / "training_dataset_manifest.json",
    v3_dir / "external_test_manifest.json",
    v3_dir / "experiment.json",
    data_dir / "multi_source_train.txt",
    data_dir / "sherlock_external_test.txt",
    external_report,
]

for path in required:
    assert path.is_file(), f"Required artifact is missing: {path}"

backup = Path("/content/ai_platform_v3_backup")
if backup.exists():
    shutil.rmtree(backup)

(backup / "model").mkdir(parents=True)
(backup / "data" / "source_files").mkdir(parents=True)
(backup / "data" / "cleaned").mkdir(parents=True)
(backup / "reports").mkdir(parents=True)

# 1. نسخة كاملة من مخرجات v3
shutil.copytree(v3_dir, backup / "model" / "artifacts")

# 2. مجموعات النصوص المستخدمة للاختبار والتدريب
shutil.copy2(
    data_dir / "multi_source_train.txt",
    backup / "data" / "cleaned" / "multi_source_train.txt",
)
shutil.copy2(
    data_dir / "sherlock_external_test.txt",
    backup / "data" / "cleaned" / "sherlock_external_test.txt",
)

# 3. حفظ سجل المصدرين مع مسارات نسبية قابلة للنقل
for filename, output_name in [
    ("multi_source_train_manifest.json", "training_manifest.json"),
    ("sherlock_external_test_manifest.json", "external_test_manifest.json"),
]:
    source_manifest_path = data_dir / filename
    manifest = json.loads(source_manifest_path.read_text(encoding="utf-8"))

    for item in manifest.get("sources", []):
        item["path"] = "data/source_files/" + Path(item["path"]).name

    (backup / "data" / output_name).write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

# 4. نسخ ملفات المصدر الأصلية، إن كانت متاحة في هذه الجلسة
source_names = ["pg11.txt", "pg84.txt", "pg1342.txt", "pg1661.txt"]
for name in source_names:
    original = source_dir / name
    assert original.is_file(), f"Original source file is missing: {original}"
    shutil.copy2(original, backup / "data" / "source_files" / name)

# 5. حفظ تقرير التقييم الخارجي وملخص المقارنة
report = json.loads(external_report.read_text(encoding="utf-8"))
(backup / "reports" / "sherlock_v3_evaluation.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

historical = report["historical_bpc"]
v3_bpc = report["v3_metrics"]["bits_per_character"]
comparison = {
    "model": "Alice Model v3 — Multi-source BPE",
    "external_dataset": "The Adventures of Sherlock Holmes",
    "validation_characters": report["v3_metrics"]["validation_characters"],
    "character_baseline_bpc": historical["Alice v1 Character"],
    "alice_v2_bpc": historical["Alice v2 BPE"],
    "alice_v3_bpc": v3_bpc,
    "relative_reduction_vs_v1_percent": (
        (historical["Alice v1 Character"] - v3_bpc)
        / historical["Alice v1 Character"] * 100
    ),
    "relative_reduction_vs_v2_percent": (
        (historical["Alice v2 BPE"] - v3_bpc)
        / historical["Alice v2 BPE"] * 100
    ),
    "unknown_characters": report["v3_metrics"]["unknown_characters"],
    "unknown_character_rate_percent": (
        report["v3_metrics"]["unknown_character_rate_percent"]
    ),
    "limitations": [
        "The model was trained for 1000 steps on three English literary books.",
        "Sherlock Holmes is an external test set, but its results have been used for model comparisons.",
        "Low BPC does not establish general conversational or reasoning ability.",
        "Research-only source status is not legal clearance for commercial use.",
    ],
}

(backup / "reports" / "comparison_summary.json").write_text(
    json.dumps(comparison, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

readme = """AI Engineering Platform — Alice Model v3 Backup

Contents
- model/artifacts/: v3 model checkpoints, BPE tokenizer, training metrics,
  training manifest, external-test manifest, and experiment configuration.
- data/cleaned/: exact training corpus and external-test text used in this run.
- data/source_files/: original Project Gutenberg source text files.
- data/: portable manifests with relative source paths.
- reports/: Sherlock external evaluation and comparison summary.

Reproduction notes
- Model architecture and training utilities are stored in the GitHub project.
- The v3 model was initialized randomly and trained on three English books.
- Sherlock Holmes was held out from training.
- The external evaluation has already been used for model comparisons; use a
  new, untouched test corpus for final model-selection claims.

Rights note
- Project Gutenberg's US public-domain classification does not guarantee
  availability in every jurisdiction. Verify local law and each source's
  terms before redistribution or commercial use.
- Source records in this experiment are marked research_only.
"""

(backup / "README.txt").write_text(readme, encoding="utf-8")

archive_path = Path("/content/ai_platform_v3_multisource_backup.zip")
if archive_path.exists():
    archive_path.unlink()

with zipfile.ZipFile(
    archive_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in sorted(backup.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(backup).as_posix())

print("=== V3 BACKUP SUMMARY ===")
print(json.dumps(comparison, ensure_ascii=False, indent=2))
print("\nArchive:", archive_path)
print("Archive size:", f"{archive_path.stat().st_size:,}", "bytes")
print("SUCCESS: v3 model, data, manifests, and reports are bundled.")

files.download(str(archive_path))

=== V3 BACKUP SUMMARY ===
{
  "model": "Alice Model v3 — Multi-source BPE",
  "external_dataset": "The Adventures of Sherlock Holmes",
  "validation_characters": 562138,
  "character_baseline_bpc": 3.0482468753495846,
  "alice_v2_bpc": 2.7867007464954323,
  "alice_v3_bpc": 2.5875072126468677,
  "relative_reduction_vs_v1_percent": 15.11490642141239,
  "relative_reduction_vs_v2_percent": 7.148005902645677,
  "unknown_characters": 38,
  "unknown_character_rate_percent": 0.006759905930572208,
  "limitations": [
    "The model was trained for 1000 steps on three English literary books.",
    "Sherlock Holmes is an external test set, but its results have been used for model comparisons.",
    "Low BPC does not establish general conversational or reasoning ability.",
    "Research-only source status is not legal clearance for commercial use."
  ]
}

Archive: /content/ai_platform_v3_multisource_backup.zip
Archive size: 7,960,130 bytes
SUCCESS: v3 model, data, manifests, and reports are bundled

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
import os
import sys
import subprocess
from pathlib import Path

repo = Path("/content/ai-engineering-platform")

# استعادة المشروع أو تحديثه إلى آخر نسخة
if not repo.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/F13342/ai-engineering-platform.git",
            str(repo),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "fetch", "origin", "main"],
        cwd=repo,
        check=True,
    )
    subprocess.run(
        ["git", "checkout", "--force", "-B", "main", "origin/main"],
        cwd=repo,
        check=True,
    )

os.chdir(repo)

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=repo,
    text=True,
).strip()

print("Commit:", commit)
assert commit == "65ee7cc", f"Expected 65ee7cc, received {commit}"

env = os.environ.copy()
env["PYTHONPATH"] = str(repo / "src") + os.pathsep + env.get("PYTHONPATH", "")

# اختبارات التدريب، بما فيها اختبارات الاستئناف الجديدة
result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q", "-ra",
        "tests/test_language_model_training.py",
    ],
    cwd=repo,
    env=env,
    capture_output=True,
    text=True,
)

print("\n--- RESUME TEST RESULTS ---")
print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Training/resume tests failed.")

print("SUCCESS: Training and resume tests passed.")

Commit: 65ee7cc

--- RESUME TEST RESULTS ---
.....F.                                                                  [100%]
=================================== FAILURES ===================================
_____________ test_resume_continues_step_count_and_optimizer_state _____________

tmp_path = PosixPath('/tmp/pytest-of-root/pytest-1/test_resume_continues_step_cou0')

    def test_resume_continues_step_count_and_optimizer_state(tmp_path):
        from dataclasses import replace
    
        corpus = tmp_path / "resume_corpus.txt"
        corpus.write_text(
            ("Alice sees the rabbit and follows it into the garden.\n" * 100),
            encoding="utf-8",
        )
        output_dir = tmp_path / "resumable_artifacts"
    
        initial_config = LanguageModelTrainingConfig(
            block_size=8,
            n_layers=1,
            n_heads=2,
            n_embd=16,
            dropout=0.0,
            batch_size=2,
            max_steps=2,
            learning_rate=1e-3

RuntimeError: Training/resume tests failed.

In [13]:
import os
import sys
import subprocess
from pathlib import Path

repo = Path("/content/ai-engineering-platform")

if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/F13342/ai-engineering-platform.git", str(repo)],
        check=True
    )

subprocess.run(["git", "fetch", "origin", "main"], cwd=repo, check=True)
subprocess.run(
    ["git", "checkout", "--force", "-B", "main", "origin/main"],
    cwd=repo,
    check=True
)

os.chdir(repo)

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=repo,
    text=True
).strip()

print("Current commit:", commit)

env = os.environ.copy()
env["PYTHONPATH"] = str(repo / "src") + os.pathsep + env.get("PYTHONPATH", "")

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-ra",
     "tests/test_language_model_training.py"],
    cwd=repo,
    env=env,
    text=True,
    capture_output=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Language model training tests failed.")

Current commit: 4fecd3c
.......                                                                  [100%]
7 passed in 2.83s



In [14]:
%cd /content/ai-engineering-platform
!PYTHONPATH=src python -m pytest -q -ra

/content
........................................................................ [ 22%]
...........F............................................................ [ 45%]
........................................................................ [ 67%]
........................................................................ [ 90%]
..............................                                           [100%]
=================================== FAILURES ===================================
________________________________ test_load_csv _________________________________

    def test_load_csv() -> None:
        """Test loading student data from a CSV file."""
        file_path = Path("data/raw/students.csv")
    
>       data = load_csv(file_path)
               ^^^^^^^^^^^^^^^^^^^

tests/test_csv_source.py:10: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 
src/ai_platform/data_engineering/sources/csv_source.py:8: in load_csv
    return pd.read_csv(file_pat

In [15]:
import os
import sys
import subprocess
from pathlib import Path

repo = Path("/content/ai-engineering-platform")

if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/F13342/ai-engineering-platform.git", str(repo)],
        check=True,
    )

subprocess.run(["git", "fetch", "origin", "main"], cwd=repo, check=True)
subprocess.run(
    ["git", "checkout", "--force", "-B", "main", "origin/main"],
    cwd=repo,
    check=True,
)

os.chdir(repo)

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=repo,
    text=True,
).strip()

print("Current commit:", commit)

env = os.environ.copy()
env["PYTHONPATH"] = str(repo / "src") + os.pathsep + env.get("PYTHONPATH", "")

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-ra"],
    cwd=repo,
    env=env,
    text=True,
    capture_output=True,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Some tests failed.")

Current commit: b5ac247
........................................................................ [ 22%]
........................................................................ [ 45%]
........................................................................ [ 67%]
........................................................................ [ 90%]
..............................                                           [100%]
318 passed in 7.55s



In [1]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

repo = Path("/content/ai-engineering-platform")

if not repo.exists():
    subprocess.run(
        ["git", "clone",
         "https://github.com/F13342/ai-engineering-platform.git",
         str(repo)],
        check=True,
    )

subprocess.run(["git", "fetch", "origin", "main"], cwd=repo, check=True)
subprocess.run(
    ["git", "checkout", "--force", "-B", "main", "origin/main"],
    cwd=repo,
    check=True,
)

os.chdir(repo)

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=repo,
    text=True,
).strip()
print("Commit:", commit)

if commit != "7c7b157":
    raise RuntimeError(f"Unexpected commit: {commit}")

# Confirm PyTorch is available in this runtime.
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("Device:", device)

# Register the project's ai-platform command without reinstalling dependencies.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"],
    cwd=repo,
    check=True,
)

if shutil.which("ai-platform") is None:
    raise RuntimeError("The ai-platform command was not found after installation.")

subprocess.run(["ai-platform", "--help"], cwd=repo, check=True)

# Create a small, reproducible smoke-test corpus.
data_dir = repo / "data" / "cli_smoke"
data_dir.mkdir(parents=True, exist_ok=True)

corpus = "Alice followed the white rabbit through the garden. " * 300
validation_text = "Alice followed the white rabbit through the garden. " * 10

corpus_path = data_dir / "train.txt"
validation_path = data_dir / "validation.txt"
corpus_path.write_text(corpus, encoding="utf-8")
validation_path.write_text(validation_text, encoding="utf-8")

output_dir = repo / "runs" / "cli_smoke"

# Train a tiny model for two steps to test CLI-to-training integration.
subprocess.run(
    [
        "ai-platform", "train",
        "--corpus", str(corpus_path),
        "--output-dir", str(output_dir),
        "--max-steps", "2",
        "--batch-size", "2",
        "--block-size", "16",
        "--n-layers", "1",
        "--n-heads", "2",
        "--n-embd", "32",
        "--eval-interval", "1",
        "--eval-batches", "1",
        "--train-fraction", "0.8",
        "--device", device,
    ],
    cwd=repo,
    check=True,
)

# Evaluate the saved best checkpoint.
subprocess.run(
    [
        "ai-platform", "evaluate",
        "--checkpoint", str(output_dir / "best.pt"),
        "--text-file", str(validation_path),
        "--device", device,
    ],
    cwd=repo,
    check=True,
)

# Generate a short continuation from the trained checkpoint.
subprocess.run(
    [
        "ai-platform", "generate",
        "--checkpoint", str(output_dir / "best.pt"),
        "--prompt", "Alice followed",
        "--max-new-tokens", "20",
        "--temperature", "1.0",
        "--device", device,
    ],
    cwd=repo,
    check=True,
)

# Verify that training artifacts were actually saved.
for name in ["best.pt", "last.pt", "tokenizer.json", "metrics.json"]:
    path = output_dir / name
    if not path.is_file():
        raise RuntimeError(f"Expected training artifact is missing: {path}")
    print(f"Verified: {name} ({path.stat().st_size:,} bytes)")

print("\nCLI train/evaluate/generate smoke test completed successfully.")

Commit: 7c7b157
PyTorch: 2.11.0+cpu
Device: cpu
Verified: best.pt (190,411 bytes)
Verified: last.pt (190,475 bytes)
Verified: tokenizer.json (300 bytes)
Verified: metrics.json (195 bytes)

CLI train/evaluate/generate smoke test completed successfully.


In [2]:
%cd /content/ai-engineering-platform
!PYTHONPATH=src python -m pytest -q -ra


/content
........................................................................ [ 22%]
........................................................................ [ 44%]
........................................................................ [ 66%]
........................................................................ [ 89%]
...................................                                      [100%]
323 passed in 18.25s


In [3]:
import json
import subprocess
from pathlib import Path
import torch

repo = Path("/content/ai-engineering-platform")
output_dir = repo / "runs" / "cli_smoke"
corpus_path = repo / "data" / "cli_smoke" / "train.txt"
checkpoint_path = output_dir / "last.pt"

# Ensure the artifacts from the previous smoke test still exist.
for path in (corpus_path, checkpoint_path):
    if not path.is_file():
        raise FileNotFoundError(
            f"Required artifact is missing: {path}. "
            "The previous Colab runtime may have been reset."
        )

# Resume the same run, extending the target from 2 to 4 total steps.
command = [
    "ai-platform", "train",
    "--corpus", str(corpus_path),
    "--output-dir", str(output_dir),
    "--resume-from", str(checkpoint_path),
    "--max-steps", "4",
    "--batch-size", "2",
    "--block-size", "16",
    "--n-layers", "1",
    "--n-heads", "2",
    "--n-embd", "32",
    "--eval-interval", "1",
    "--eval-batches", "1",
    "--train-fraction", "0.8",
    "--device", "cpu",
]

subprocess.run(command, cwd=repo, check=True)

# Verify the full training history and final checkpoint.
metrics = json.loads(
    (output_dir / "metrics.json").read_text(encoding="utf-8")
)
steps = [item["step"] for item in metrics]

checkpoint = torch.load(
    output_dir / "last.pt",
    map_location="cpu",
    weights_only=True,
)

assert steps == [1, 2, 3, 4], f"Unexpected training history: {steps}"
assert checkpoint["step"] == 4, (
    f"Expected checkpoint step 4, got {checkpoint['step']}"
)

print("Training history:", steps)
print("Final checkpoint step:", checkpoint["step"])
print("CLI resume test passed.")

Training history: [1, 2, 3, 4]
Final checkpoint step: 4
CLI resume test passed.


## 2. ????? ?????? ?? ?????

?????? ???? ???????? ????????? ?????? ?????? ??? ?? ??????? ???. ?? ?????? ??? ???????? ?????? ???? ??????? ?? ??????? ??????? ?????? ???????.

In [ ]:
from ai_platform.machine_learning.language_model.training import (
    LanguageModelTrainingConfig,
    train_language_model,
    load_language_model,
    generate_text,
)

corpus_text = (
    'AI engineering platform trains its own language model from random initialization. '
    'A tokenizer converts characters into token IDs. A transformer learns to predict '
    'the next character from previous characters. The training loop measures loss, '
    'updates model parameters, and saves checkpoints. '
) * 120

corpus_path = Path('/content/model_lab_smoke_corpus.txt')
corpus_path.write_text(corpus_text, encoding='utf-8')

output_dir = Path('/content/model_lab_smoke_artifacts')
config = LanguageModelTrainingConfig(
    block_size=32,
    n_layers=2,
    n_heads=2,
    n_embd=64,
    dropout=0.1,
    batch_size=16,
    max_steps=30,
    learning_rate=3e-4,
    eval_interval=10,
    eval_batches=3,
    train_fraction=0.9,
    seed=42,
    device='auto',
)

result = train_language_model(corpus_path, output_dir, config)
print('\nTraining result:')
print(result)


## 3. ????? ????? ??????? ?????? ??

????? ???? checkpoint ??????? ?? ??????? ??????? ?????? ?? ??????? ???????? ??? ?????? ???? ???????.

In [ ]:
checkpoint_path = output_dir / 'best.pt'
model, tokenizer, checkpoint = load_language_model(checkpoint_path, device='auto')

print('Checkpoint step:', checkpoint['step'])
print('Vocabulary size:', tokenizer.vocab_size)
print('Model is in evaluation mode:', not model.training)
print('\nGenerated sample:')
print(generate_text(
    checkpoint_path,
    prompt='AI engineering',
    max_new_tokens=120,
    temperature=0.8,
    top_k=10,
    device='auto',
))

assert (output_dir / 'best.pt').is_file()
assert (output_dir / 'last.pt').is_file()
assert (output_dir / 'tokenizer.json').is_file()
assert (output_dir / 'metrics.json').is_file()
print('\nSUCCESS: Training, checkpoint files, reload, and generation completed.')


## 4. ????? ????? ???????

???? ??????? ???????? ????? ?? ??????? ????????? ??????? ???????. ??? ??????? ??? ??????? ???????? ???.

In [ ]:
import shutil
from google.colab import files

archive_path = shutil.make_archive(
    '/content/model_lab_smoke_artifacts',
    'zip',
    root_dir=output_dir,
)
files.download(archive_path)
